### Composite Isolated and Organized convective intensity interregional comparisons for tropical West Atlantic (CPEX, CPEX-AW), East Atlantic (CPEX-CV), and Northwest Pacific (CAMP2Ex)

In [ ]:
import os
import sys
import math
import h5py
import xarray as xr
import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm  #to get python's normal library of colormaps
import matplotlib.colors as mplc
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

import cartopy.crs as ccrs
import cartopy.feature as cfeature
#from cartopy.util import add_cyclic_point
#from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

from datetime import datetime
from datetime import timedelta

import metpy.calc as mpcalc
import metpy.plots as mplots
from metpy.units import units

import scipy.stats
from scipy.stats import norm
import scipy.signal as sig

from PIL import Image
import icartt            #needed to read .ict files

import time

# import warnings
# warnings.filterwarnings("ignore")  #hides "MatplotlibDeprecationWarning" with pcolormesh

# tstart = time.time()


### Convective Regions Only

In [ ]:
#CFADs (with and without convective-stratiform partitioning)
#adapted from CFAD_AGUpapers_only_pseudonadir_ray_and_ECCO.py

matplotlib.rcParams['font.family'] = 'arial'
# matplotlib.rcParams['axes.labelsize'] = 14
# matplotlib.rcParams['axes.titlesize'] = 14
# matplotlib.rcParams['xtick.labelsize'] = 12
# matplotlib.rcParams['ytick.labelsize'] = 12
matplotlib.rcParams['legend.fontsize'] = 24
#matplotlib.rcParams['legend.facecolor'] = 'w'

#create the dictionary of cases for which you want to plot a total CFAD and create the height/reflectivity bin edges

# #CAMP2Ex cases (Cases 1-14 are Isolated; Cases 15-19 are Organized)
# case1_dict = {1: ['20190829', '234500','040000']}      #Good APR-3 data coverage
# case2_dict = {2: ['20190829', '041500','051000']}      #Meh APR-3 data coverage, probably don’t use for CFADs
# case3_dict = {3: ['20190831', '000000', '001000']}     #No APR-3 data coverage, techincally case starts at 23:50 UTC on 20190830, but no APR-3 data files until 20190831
# case4_dict = {4: ['20190831', '050000', '055000']}     #Meh APR-3 data coverage, probably don’t use for CFADs
# case5_dict = {5: ['20190904', '014500', '050000' ]}    #No APR-3 data coverage
# case6_dict = {6: ['20190904', '050000', '060500']}     #No APR-3 data coverage
# case7_dict = {7: ['20190904', '060500', '071500']}     #No APR-3 data coverage
# case8_dict = {8: ['20190907', '062000', '071000']}     #Good APR-3 data coverage
# case9_dict = {'9a': ['20190909', '003500', '005500'], '9b': ['20190909', '023000', '024500']}   #No APR-3 data coverage
# case10_dict = {10: ['20190909', '005500', '022500']}   #Good APR-3 data coverage
# case11_dict = {11: ['20191001', '220000', '224500']}   #No APR-3 data coverage
# case12_dict = {12: ['20191002', '053000', '061500']}   #No APR-3 data coverage
# case13_dict = {13: ['20191003', '231000', '004500']}   #No APR-3 data coverage
# case14_dict = {14: ['20191004', '004500', '010500']}   #No APR-3 data coverage
# case15_dict = {'15a': ['20190907', '002500', '024000'], '15b': ['20190907', '031000', '040500']}   #Good APR-3 data coverage and it is through the center of the storm (though not at its most intense)
# case16_dict = {'16a': ['20190907', '024500', '031000'], '16b': ['20190907', '040500', '044500'], '16c': ['20190907', '071000', '074000']}   #Little APR-3 data coverage (definitely not good enough for CFADs), even though the flight goes right through the growing storm; APR-3 reflectivity from 02:45:00 – 03:10:00 UTC is from the infancy of the storm (and maybe not the storm at all) and so is not really representative of the storm
# case17_dict = {'17a': ['20190915', '223500', '230500'], '17b': ['20190915', '031000', '044500']}   #Good APR-3 data coverage and it is through the center of the storm
# case18_dict = {18: ['20190917', '014500', '061500']}   #Meh good APR-3 data coverage, but coverage really only on the outer edge of the convection (nowhere close to the center/strongest parts of the storm)
# case19_dict = {19: ['20191005', '023000', '065000']}   #Bad APR-3 data coverage, and coverage only on the far outer edge of the convection after 6 UTC (nowhere close to center/strongest parts of the storm)

nwpac_isolated_case_dict = {1: ['20190829', '234500','040000'], 8: ['20190907', '062000', '071000'], 10: ['20190909', '005500', '022500']}   #cases 1, 8, 10 above
nwpac_organized_case_dict = {'15a': ['20190907', '002500', '024000'], '15b': ['20190907', '031000', '040500'], 
                             '17a': ['20190915', '223500', '230500'], '17b': ['20190915', '031000', '044500']}   #cases 15, 17 above

#cases from nwpac_xxx_case_dict above that were at least partly collected during growing or mature lifecycle stages
nwpac_isolated_case_dict_growing_or_mature = {1: ['20190829', '234500','040000'], 10: ['20190909', '005500', '022500']}   #cases 1, 10 above
nwpac_organized_case_dict_growing_or_mature = {'15a': ['20190907', '002500', '024000'], '15b': ['20190907', '031000', '040500']}   #case 15 above

# #CPEX(-AW) cases
# case1_dict = {1: ['20170610','194655','221900']}
# case2_dict = {2: ['20170624','180000','194800']}
# case3_dict = {3: ['20170624','201200','220000']}
# case4_dict = {4: ['20170615', '184840', '205000']}
# case5_dict = {5: ['20170616', '182451', '220600']}
# case6_dict = {6: ['20170601', '175849', '220700']}
# case7_dict = {7: ['20170606', '185211', '215000']}
# case8_dict = {8: ['20170617', '184650', '220000']}
# case13_dict = {13: ['20170611', '180100', '203400']}
# case14_dict = {14: ['20210821', '221800', '234145']}
# case16_dict = {16: ['20210824', '181545', '195745']}

watl_isolated_case_dict = {1: ['20170610','194655','221900'], 2: ['20170624','180000','194800'], 3: ['20170624','201200','220000']}   #cases 1, 2, 3 above
watl_organized_case_dict = {4: ['20170615', '184840', '205000'], 5: ['20170616', '182451', '220600'], 6: ['20170601', '175849', '220700'],
                            7: ['20170606', '185211', '215000'], 8: ['20170617', '184650', '220000'], 13: ['20170611', '180100', '203400'],
                            14: ['20210821', '221800', '234145'], 16: ['20210824', '181545', '195745']}   #cases 4, 5, 6, 7, 8, 13, 14, 16 above

#cases from watl_xxx_case_dict above that were at least partly collected during growing or mature lifecycle stages (turns out to be all the cases from above)
watl_isolated_case_dict_growing_or_mature = {1: ['20170610','194655','221900'], 2: ['20170624','180000','194800'], 3: ['20170624','201200','220000']}   #cases 1, 2, 3 above
watl_organized_case_dict_growing_or_mature = {4: ['20170615', '184840', '205000'], 5: ['20170616', '182451', '220600'], 6: ['20170601', '175849', '220700'],
                                              7: ['20170606', '185211', '215000'], 8: ['20170617', '184650', '220000'], 13: ['20170611', '180100', '203400'],
                                              14: ['20210821', '221800', '234145'], 16: ['20210824', '181545', '195745']}   #cases 4, 5, 6, 7, 8, 13, 14, 16 above

# #CPEX-CV cases (Cases 1-7 are Isolated; Cases 8-22 are Organized (ignore Case 19 (TC)); Cases 23-24 are Scattered)
# case1_dict = {1: ['20220909','161000','171000']}
# case2_dict = {2: ['20220909','173500','191000']}
# case3_dict = {3: ['20220910','190500','194200']}
# case4_dict = {4: ['20220910', '202500', '204500']}
# case5_dict = {5: ['20220916', '155500', '163500']}
# case6_dict = {6: ['20220920', '071000', '073500']}
# case7_dict = {7: ['20220920', '083000', '090000']}
# case8_dict = {8: ['20220906', '110000', '120000']}
# case9_dict = {'9a': ['20220906', '133000', '143000'], '9b': ['20220906', '153000', '161000']}
# case10_dict = {10: ['20220906', '161000', '180000']}
# case11_dict = {11: ['20220907', '130000', '134500']}
# case12_dict = {'12a': ['20220907', '134500', '151300'], '12b': ['20220907', '161800', '174500']}
# case13_dict = {13: ['20220907', '151300', '161800']}
# case14_dict = {'14a': ['20220914', '101000', '115500'], '14b': ['20220914', '134000', '142800']}
# case15_dict = {'15a': ['20220914','115500','131000'], '15b': ['20220914','142800','164500']}
# case16_dict = {16: ['20220916','143000','154000']}
# case17_dict = {17: ['20220916','164000','183500']}
# case18_dict = {'18a': ['20220922', '054000', '061500'], '18b': ['20220922', '063500', '073600'], '18c': ['20220922', '080000', '083000']}
# case20_dict = {20: ['20220926', '072000', '111500']}
# case21_dict = {21: ['20220929', '103500', '134500']}
# case22_dict = {22: ['20220930', '134800', '143000']}

eatl_isolated_case_dict = {1: ['20220909','161000','171000'], 2: ['20220909','173500','191000'], 3: ['20220910','190500','194200'],
                           4: ['20220910', '202500', '204500'], 5: ['20220916', '155500', '163500'], 6: ['20220920', '071000', '073500'],
                           7: ['20220920', '083000', '090000']}   #cases 1, 2, 3, 4, 5, 6, 7 above
eatl_organized_case_dict = {8: ['20220906', '110000', '120000'], '9a': ['20220906', '133000', '143000'], '9b': ['20220906', '153000', '161000'],
                            10: ['20220906', '161000', '180000'], 11: ['20220907', '130000', '134500'], '12a': ['20220907', '134500', '151300'],
                            '12b': ['20220907', '161800', '174500'], 13: ['20220907', '151300', '161800'], '14a': ['20220914', '101000', '115500'],
                            '14b': ['20220914', '134000', '142800'], '15a': ['20220914','115500','131000'], '15b': ['20220914','142800','164500'],
                            16: ['20220916','143000','154000'], 17: ['20220916','164000','183500'], '18a': ['20220922', '054000', '061500'],
                            '18b': ['20220922', '063500', '073600'], '18c': ['20220922', '080000', '083000'], 20: ['20220926', '072000', '111500'],
                            21: ['20220929', '103500', '134500'], 22: ['20220930', '134800', '143000']}   #cases 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22 above

#cases from eatl_xxx_case_dict above that were at least partly collected during growing or mature lifecycle stages
eatl_isolated_case_dict_growing_or_mature = {1: ['20220909','161000','171000'], 3: ['20220910','190500','194200'], 7: ['20220920', '083000', '090000']}   #cases 1, 3, 7 above
eatl_organized_case_dict_growing_or_mature = {8: ['20220906', '110000', '120000'], '9a': ['20220906', '133000', '143000'], '9b': ['20220906', '153000', '161000'],
                                              10: ['20220906', '161000', '180000'], '12a': ['20220907', '134500', '151300'], '12b': ['20220907', '161800', '174500'],
                                              13: ['20220907', '151300', '161800'], '15a': ['20220914','115500','131000'], '15b': ['20220914','142800','164500'],
                                              16: ['20220916','143000','154000'], 17: ['20220916','164000','183500'], '18a': ['20220922', '054000', '061500'],
                                              '18b': ['20220922', '063500', '073600'], '18c': ['20220922', '080000', '083000'], 20: ['20220926', '072000', '111500'],
                                              21: ['20220929', '103500', '134500'], 22: ['20220930', '134800', '143000']}   #cases 8, 9, 10, 12, 13, 15, 16, 17, 18, 20, 21, 22 above


height_edges = np.arange(1500, 8001, 500)
dbz_edges = np.arange(-20, 70.1, 5)
vel_edges = np.arange(-25, 25.1, 2)
# dbz_edges = np.arange(-20, 60.1, 5)
# vel_edges = np.arange(-13, 15.1, 2)


def CFAD(case_dict, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True):
    
    """"Calculate and plot Ku-band and Doppler Velocity CFAD 2-D arrays for 
        the given cases and their respective time ranges
    
    Parameters
    ----------
    case_dict:  dictionary of cases for which to calculate the total CFAD; 
                keys should be case numbers;
                values should be 3-element lists of case date and start/end times, 
                with date strings formatted as YYYYMMDD and time strings formatted as HHMMSS
    
    height_bin_edges:  a 1-D array of height [m] bin edges used to create the 2-D CFAD array/histogram    

    dbz_bin_edges:  a 1-D array of reflectivity [dBZ] bin edges used to create the 2-D CFAD array/histogram 

    vel_bin_edges:  a 1-D array of Doppler velocity [m/s] bin edges used to create the 2-D CFAD array/histogram       
                        
    normalized:  True/False; determines whether to normalize the CFAD array by maximum bin at any level 
                 (method from Zagrodnik et al., 2019) and create an additional, normalized CFAD plot
                        
    return:  2-D (normalized) CFAD arrays and plots """

    assert type(case_dict) == dict, "case_dict must be a dictionary"
    
    total_apr_profiles = 0 
    apr_profile_roll10 = 0
    new_ku_array = True
    new_vel_array = True
    
    for key in case_dict:
        print ('Processing Case {}...'.format(key))
        
        first_good_Ku_file_index = 0    #used to determine first usable Ku-band file for CAMP2Ex cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
        
        #grab the case's date and start/end times from the dictionary
        assert type(case_dict[key]) == list, "A key's values must be a list of date, start time, and end time" 
        desired_date = case_dict[key][0]
        if desired_date[:4] == '2019' and (key == 1 or key == 2 or key == 13 or key == '17b'):
            # #if a CAMP2Ex case spans across multiple days or the time range is fully the day after its science flight start date
            # case1_dict = {1: ['20190829',check code'234500','040000']}
            # case2_dict = {2: ['20190829',check code'041500','051000']}
            # case13_dict = {13: ['20191003', check code'231000', '004500']}
            # case17_dict = {'17a': ['20190915','223500','230500'], '17b': ['20190915',check code'031000','044500']}
            desired_date_next = datetime.strftime(datetime.strptime(desired_date, '%Y%m%d') + timedelta(days = 1), '%Y%m%d')
        else:
            desired_date_next = desired_date + ''
        start_time = case_dict[key][1]
        end_time = case_dict[key][2]
        #assert start_time < end_time, "Start time must precede end time in a key's list"
        
        #grab the ECCO-V convective-stratiform classification file for the given date
        use_file_list = []
        for file in os.listdir(os.path.join(os.getcwd(), 'ECCO-V_output', 'ECCO-V_classification_1D')):
            if desired_date in file:
                use_file_list.append(file)
        assert len(use_file_list) == 1, "Found either 0 or multiple ECCO-V files for the given date"
        use_file = use_file_list[0]
        
        ecco_filepath = os.path.join(os.getcwd(), 'ECCO-V_output', 'ECCO-V_classification_1D', use_file)
        ecco_df = pd.read_csv(ecco_filepath, sep = ',', dtype = str)
        ecco_df['Full_Datetime'] = pd.to_datetime(ecco_df['Year'].str.zfill(4) + ecco_df['Month'].str.zfill(2) + ecco_df['Day'].str.zfill(2) + ecco_df['Hour'].str.zfill(2) + ecco_df['Minute'].str.zfill(2) + ecco_df['Second'].astype(float).round().astype(int).astype(str), format = '%Y%m%d%H%M%S')
        
        #find the APR files of interest (for the desired date and time ranges)
        apr_folder = os.path.join(desired_date, 'APR_files')
        apr_file_list = sorted(os.listdir(apr_folder))
        
        #angles for each of the 24 rays in a given scan (used for ray adjustment; index order goes from left to right in the scan when looking ahead in the direction that the aircraft is headed)
        if desired_date[:4] == '2017':
            ray_angles = np.linspace(-25,25,24)[:-1]  #in degrees; omits 24th ray, which doesn't have data for Ku/Ka bands
        
            #find the list of APR files that have data for the desired time range
            apr_files_use = []
            first_file = 'blank'
            for file in apr_file_list:        #sorted() makes sure the code goes through the files in alphabetical order
                if file[0:3] == '.DS':         #delete possible .DS_Store files
                    os.remove(os.path.join(apr_folder, file))
                elif file[22:28] <= start_time:
                    first_file = file     #first_file will always be the file immediately before (or equal to) range_start
                elif file[22:28] >= end_time:
                    continue
                else:
                    if (first_file not in apr_files_use) and (first_file != 'blank'):
                        apr_files_use.append(first_file)
                    apr_files_use.append(file)
                    
            if apr_files_use == []:             #accounts for if start_time is greater than all of the file times, but still within the last file's time range; also accounts for start/end times equaling the times of adjacent files
                apr_files_use.append(first_file)
                
        elif desired_date[:4] == '2021':
            ray_angles = np.linspace(-25,25,25)
            
            #create a list of all the given day's desired range's APR files:    #for APR_plots.py
            start_time1 = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
            end_time1 = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
            
            for x in os.listdir(apr_folder):
                if x[0:3] == '.DS':         #delete hidden .DS_Store files if they come up (will show up if you delete a file)
                    os.remove(os.path.join(apr_folder, x))
            
            #find the starting APR file in apr_folder
            first_file_index = None       
            for i, x in enumerate(apr_file_list):
                file_start_time = datetime.strptime(x[13:21] + x[22:28], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
            
                if start_time1 <= file_start_time:  #if start_time1 is before the APR file start time and not within any previous APR file's time ranges
                    first_file_index = i
                    break
                elif (start_time1 >= file_start_time) and (start_time1 < file_end_time):
                    first_file_index = i
                    break
                else:
                    continue
            if first_file_index == None:
                sys.exit('Requested start_time is beyond all available APR files')
                
            #find the ending APR file in apr_folder
            last_file_index = None       
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[13:21] + x[22:28], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
            
                if end_time1 <= file_start_time:  #if end_time1 is before the APR file start time and not within any previous APR file's time ranges
                    last_file_index = i - 1
                    break
                elif (end_time1 > file_start_time) and (end_time1 <= file_end_time):
                    last_file_index = i
                    break
                else:
                    continue
            if last_file_index == None:  #the end_time is after all available APR files
                last_file_index = len(apr_file_list) - 1  #the last available APR file's index
            if last_file_index == -1:
                sys.exit('Requested end_time is before all available APR files')
            
            apr_files_use = apr_file_list[first_file_index:last_file_index + 1]
        
        elif desired_date[:4] == '2022':
            ray_angles = np.linspace(-25,25,25)
            
            #create a list of all the given day's desired range's APR files:    #for APR_plots.py
            start_time1 = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
            end_time1 = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
            
            #create a list of all the given day's desired range's APR files:
            for x in os.listdir(apr_folder):
                if x[0:3] == '.DS':         #delete hidden .DS_Store files if they come up (will show up if you delete a file)
                    os.remove(os.path.join(apr_folder, x))
            
            #find the starting APR file in apr_folder
            first_file_index = None 
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[46:54] + x[55:61], '%Y%m%d%H%M%S')
            
                if start_time1 <= file_start_time:  #if start_time is before the APR file start time and not within any previous APR file's time ranges
                    first_file_index = i
                    break
                elif (start_time1 >= file_start_time) and (start_time1 < file_end_time):
                    first_file_index = i
                    break
                else:
                    continue
            if first_file_index == None:
                sys.exit('Requested start_time is beyond all available APR files')
                
            #find the ending APR file in apr_folder
            last_file_index = None    
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[46:54] + x[55:61], '%Y%m%d%H%M%S')
            
                if end_time1 <= file_start_time:  #if end_time is before the APR file start time and not within any previous APR file's time ranges
                    last_file_index = i - 1
                    break
                elif (end_time1 > file_start_time) and (end_time1 <= file_end_time):
                    last_file_index = i
                    break
                else:
                    continue
            if last_file_index == None:  #the end_time is after all available APR file ranges
                last_file_index = len(apr_file_list) - 1  #the last available APR file's index
            if last_file_index == -1:
                sys.exit('Requested end_time is before all available APR files')
            
            apr_files_use = apr_file_list[first_file_index:last_file_index + 1]
            
        elif desired_date[:4] == '2019':
            ray_angles = np.linspace(-25,25,25)
            
            #create a list of all the given day's desired range's APR files:    #for APR_plots.py
            if key == 2 or key == '17b':
                # #if a case spans across multiple days or the time range is fully the day after its science flight start date
                # case1_dict = {1: ['20190829',check code'234500','040000']}
                # case2_dict = {2: ['20190829',check code'041500','051000']}
                # case13_dict = {13: ['20191003', check code'231000', '004500']}
                # case17_dict = {'17a': ['20190915','223500','230500'], '17b': ['20190915',check code'031000','044500']}
                start_time1 = datetime.strptime(desired_date_next + start_time, '%Y%m%d%H%M%S')
            else:
                start_time1 = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
            
            end_time1 = datetime.strptime(desired_date_next + end_time, '%Y%m%d%H%M%S')
                
            #print (f'Case {key} start time: {start_time1}')  #sanity check to make sure the date ranges are correct
            #print (f'Case {key} end time: {end_time1}')      #sanity check to make sure the date ranges are correct
            
            #create a list of all the given day's desired range's APR files:
            for x in os.listdir(apr_folder):
                if x[0:3] == '.DS':         #delete hidden .DS_Store files if they come up (will show up if you delete a file)
                    os.remove(os.path.join(apr_folder, x))
            
            #find the starting APR file in apr_folder
            first_file_index = None
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[35:43] + x[44:50], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[52:60] + x[61:67], '%Y%m%d%H%M%S')

                if start_time1 <= file_start_time:  #if start_time is before the APR file start time and not within any previous APR file's time ranges
                    first_file_index = i
                    break
                elif (start_time1 >= file_start_time) and (start_time1 < file_end_time):
                    first_file_index = i
                    break
                else:
                    continue
            if first_file_index == None:
                sys.exit('Requested start_time is after all available APR files')
                
            #find the ending APR file in apr_folder
            last_file_index = None 
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):   
                file_start_time = datetime.strptime(x[35:43] + x[44:50], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[52:60] + x[61:67], '%Y%m%d%H%M%S')
            
                if end_time1 <= file_start_time:  #if end_time is before the APR file start time and not within any previous APR file's time ranges
                    last_file_index = i - 1
                    break
                elif (end_time1 > file_start_time) and (end_time1 <= file_end_time):
                    last_file_index = i
                    break
                else:
                    continue
            if last_file_index == None:  #the end_time is after all available APR file ranges
                last_file_index = len(sorted(os.listdir(apr_folder))) - 1  #the last available APR file's index
            if last_file_index == -1:
                sys.exit('Requested end_time is before all available APR files')
                
            apr_files_use = apr_file_list[first_file_index:last_file_index + 1]
        
        else:
            sys.exit('Not a CPEX or CPEX-AW or CPEX-CV or CAMP2Ex case')
    
        #print (apr_files_use)  #sanity check to make sure you grabbed the right files


        for apr_filepath in apr_files_use:
            
            if desired_date[:4] == '2017' or desired_date[:4] == '2021':    #CPEX(-AW)
            
                #Low resolution ('lores') radar variables in APR hdf files
                ku_band = 'zhh14' #Ku-band reflectivity
                vel = 'vel14c' #mean Doppler Velocity dealiased and from Ku&Ka band
            
                apr_file = h5py.File(os.path.join(apr_folder, apr_filepath), 'r')
                
                if ('lores' in apr_file.keys()) and (ku_band in apr_file['lores'].keys()):
                    
                    try:   #some CPEX-AW APR files have corrupted Ku-band data; if so, skip the Ku-band for 
                           #that file (corrupted: "OSError: Can't read data (inflate() failed)")
                        ku_data = apr_file['lores'][ku_band][:]
                    except:
                        apr_file.close()
                        continue  #both Ku-band and velocity CFADs rely on Ku-band data availability
    
                    try:   #some CPEX-AW APR files have corrupted velocity data; if so, skip the Ku-band for 
                           #that file (corrupted: "OSError: Can't read data (inflate() failed)")
                        vel_data = apr_file['lores'][vel][:]
                        vel_good = True
                    except:
                        vel_good = False                    
                
                    #grab the radar variables of interest
                    time = apr_file['lores']['scantime'][:]
                    alt3d = apr_file['lores']['alt3D'][:]
                    roll = apr_file['lores']['roll'][:]
                    
                    if apr_filepath == apr_files_use[0] or apr_filepath == apr_files_use[-1]:
                    #the complete APR file time range may not need to be used, so need to locate the closest time (and corresponding index) to the desired start/end time
                        
                        #Convert APR times to datetimes
                        time_dates = np.empty(time.shape[1], dtype=object)
                        for j in np.arange(0, time.shape[1]):
                            #tmp = datetime(time[12,j])
                            tmp = datetime.utcfromtimestamp(time[12,j])  #12 could be any ray, as it is the ray number and each ray of a given scan has the same time
                            time_dates[j] = tmp
                        
                        unique_apr_times = time_dates  #all the times in the given APR file
                        
                        if len(apr_files_use) == 1:  #i.e. apr_filepath == apr_files_use[0] and apr_filepath == apr_files_use[-1]
                        
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                        
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                                    
                        elif apr_filepath == apr_files_use[0]:  #i.e. the first APR file in the apr_files_use list
                            end_time_idx = time.shape[1] - 1
                            
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                                    
                        else:  #apr_filepath == apr_files_use[-1]  #i.e. the last APR file in the apr_files_use list
                            start_time_idx = 0
                            
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))                
                            
                    else:  #the entire APR file is within the desired time range, so set the start/end indices to the first/last indices of the APR file
                        start_time_idx = 0
                        end_time_idx = time.shape[1] - 1
                            
                    #numpy.histogram2d(dBZ, z) (Zagrodnik et al., 2019) CFAD method:

                    #loop through the given APR file’s valid times/scans/profiles
                    for time_idx in range(start_time_idx, end_time_idx + 1):          #time_idx represents the scan number
                    
                        profile_time = datetime.utcfromtimestamp(time[12, time_idx])  #12 could be any ray, as it is the ray number and each ray of a given scan has the same time
                        ecco_ip = np.argmin(abs(ecco_df['Full_Datetime'] - profile_time))  #abs() necessary to properly subtract datetime objects
                                                                                            #returns the integer position of the minimum value
                        #print (ecco_df['Full_Datetime'].iloc[ecco_index])           #to check that the correct time was chosen
                        classification = ecco_df['Classification'].iloc[ecco_ip]     #grab the classification of the ECCO-V time closest to the given profile time (may/should be the exact time)
                        
                        if classification not in ['0', '1', '2']:
                            raise ValueError(f'{classification} is not a valid ECCO-V classification.')
                    
                        #skip profiles with an ECCO-V classification of "no classification" (0) or "stratiform" (1)
                        #i.e., only use "convective" (2) profiles
                        if classification == '0' or classification == '1':
                            continue
                        
                        # #skip profiles with an ECCO-V classification of "no classification" (0) or "convective" (2)
                        # #i.e., only use "stratiform" (1) profiles
                        # if classification == '0' or classification == '2':
                        #     continue
                        
                        total_apr_profiles += 1
                        
                        #choose the "nadir" ray factoring in aircraft roll
                        ac_roll = np.nanmean(roll[:,time_idx])  #roll varies slightly w/ray, so take the average roll value for a given scan and use that for ray adjustment
                        ray_use = np.argmin(np.abs(ray_angles - ac_roll))  #the index of the ray whose angle is closest to that of ac_roll
                        
                        if abs(ac_roll) >= 10:
                            apr_profile_roll10 += 1
                        
                        prof_height = alt3d[:,ray_use,time_idx]
                        prof_dbz = ku_data[:,ray_use,time_idx]
                        
                        #grab the height/dbz data above 1.5km
                        idx_1500m = np.argmin(np.abs(prof_height - 1500))
                        prof_height_above1500m = np.flip(prof_height[:idx_1500m + 1])  #height profile above 1.5km, in ascending order
                        prof_dbz_above1500m = np.flip(prof_dbz[:idx_1500m + 1])  #Ku-band profile above 1.5km, in ascending order
                        
                        #store all the profile's height/dbz data in long 1-D concatenated arrays
                        if new_ku_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                            height_concat = prof_height.copy()
                            dbz_concat = prof_dbz.copy()
                            
                            #create a DataFrame of values above 1.5km, in order to calculate meadian/quartile reflectivity profile for the given case
                                #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                    #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile     
                            height_profiles_above1500m_df = pd.DataFrame(prof_height_above1500m)  #height profile from 1500m to top of profile
                            dbz_profiles_above1500m_df = pd.DataFrame(prof_dbz_above1500m)        #dbz profile from 1500m to top of profile
                            new_ku_array = False
                        else:
                            height_concat = np.concatenate((height_concat, prof_height))
                            dbz_concat = np.concatenate((dbz_concat, prof_dbz))
                            
                            #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
                            height_profiles_above1500m_df = pd.concat((height_profiles_above1500m_df, pd.Series(prof_height_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                            dbz_profiles_above1500m_df = pd.concat((dbz_profiles_above1500m_df, pd.Series(prof_dbz_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                        
                        # if vel_good:
                        #     prof_vel = vel_data[:,ray_use,time_idx]
                            
                        #     #store all the profile's velocity data in long 1-D concatenated arrays if 
                        #     #the profile has Ku data > 0 dBZ above 1.5km (i.e. omits clear profiles)
                            
                        #     if np.nanmax(prof_dbz_above1500m) > 0:
                        #         if new_vel_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                        #             vel_concat = prof_vel.copy()
                        #             height_vel_concat = prof_height.copy()
                        #             new_vel_array = False
                        #         else:
                        #             vel_concat = np.concatenate((vel_concat, prof_vel))
                        #             height_vel_concat = np.concatenate((height_vel_concat, prof_height))
                        #     else:  #clear profiles (which may have noisy velocity data) are omitted from the velocity CFAD
                        #         pass
                        # else:
                        #     pass
                else:
                    pass
                    
                apr_file.close()
                
            elif desired_date[:4] == '2022':    #CPEX-CV
            
                ku_band = 'lores_zhh14' #Ku-band reflectivity
                vel = 'lores_vel14c' #Mean Doppler Velocity from Ku-band (surface Doppler velocity is subtracted and free of aliasing)
            
                apr_file = xr.open_dataset(os.path.join(apr_folder, apr_filepath))
                
                if ku_band in apr_file.keys():
                    
                    try:   #some APR files, at least in the preliminary data, have corrupted Ku-band data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        ku_data = apr_file[ku_band][:]
                    except:
                        first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                        apr_file.close()
                        continue  #both Ku-band and velocity CFADs rely on Ku-band data availability
    
                    try:   #some APR files, at least in the preliminary data, have corrupted velocity data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        vel_data = apr_file[vel][:]
                        vel_good = True
                    except:
                        vel_good = False                    
                
                    #grab the radar variables of interest
                    time = apr_file['time'][:]          #For 'lores': Time of scan, in seconds since midnight UTC of [YYYY-mm-DD]
                    alt3d = apr_file['lores_alt3D'][:]
                    roll = apr_file['lores_roll'][:]
                    
                    if apr_filepath == apr_files_use[first_good_Ku_file_index] or apr_filepath == apr_files_use[-1]:
                    #the complete APR file time range may not need to be used, so need to locate the closest time (and corresponding index) to the desired start/end time
                        
                        #Convert APR times to datetimes
                        time_dates = np.empty(time.shape, dtype=object)
                        for i in np.arange(0, time.shape[0]):
                            #hour, second automatically revert to midnight (hour = 0, seconds = 0) for '%Y%m%d'
                            time_dates[i] = datetime.strptime(desired_date, '%Y%m%d') + timedelta(seconds = float(time[i].values))
                        
                        unique_apr_times = time_dates[:]  #all the times in the given APR file
                        
                        if len(apr_files_use) == 1:  #i.e. apr_filepath == apr_files_use[first_good_Ku_file_index] and apr_filepath == apr_files_use[-1]
                        
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                        
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                                    
                        elif apr_filepath == apr_files_use[first_good_Ku_file_index]:  #i.e. the first APR file in the apr_files_use list
                            end_time_idx = time.shape[0] - 1
                            
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                                    
                        else:  #apr_filepath == apr_files_use[-1]  #i.e. the last APR file in the apr_files_use list
                            start_time_idx = 0
                            
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))                
                            
                    else:  #the entire APR file is within the desired time range, so set the start/end indices to the first/last indices of the APR file
                        start_time_idx = 0
                        end_time_idx = time.shape[0] - 1
                            
                    #numpy.histogram2d(dBZ, z) (Zagrodnik et al., 2019) CFAD method:
                    
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                       
                    
                    #loop through the given APR file’s valid times/scans/profiles
                    for time_idx in range(start_time_idx, end_time_idx + 1):          #time_idx represents the scan number
                        
                        profile_time = datetime.strptime(desired_date, '%Y%m%d') + timedelta(seconds = float(time[time_idx].values))
                        ecco_ip = np.argmin(abs(ecco_df['Full_Datetime'] - profile_time))  #abs() necessary to properly subtract datetime objects
                                                                                            #returns the integer position of the minimum value
                        #print (ecco_df['Full_Datetime'].iloc[ecco_index])           #to check that the correct time was chosen
                        classification = ecco_df['Classification'].iloc[ecco_ip]     #grab the classification of the ECCO-V time closest to the given profile time (may/should be the exact time)
                        
                        if classification not in ['0', '1', '2']:
                            raise ValueError(f'{classification} is not a valid ECCO-V classification.')
                    
                        #skip profiles with an ECCO-V classification of "no classification" (0) or "stratiform" (1)
                        #(i.e., only use "convective" (2) profiles)
                        if classification == '0' or classification == '1':
                            continue
                        
                        # #skip profiles with an ECCO-V classification of "no classification" (0) or "convective" (2)
                        # #(i.e., only use "stratiform" (1) profiles)
                        # if classification == '0' or classification == '2':
                        #     continue
                        
                        total_apr_profiles += 1
                        
                        #choose the "nadir" ray factoring in aircraft roll
                        ac_roll = np.nanmean(roll[time_idx,:])  #roll varies slightly w/ray, so take the average roll value for a given scan and use that for ray adjustment
                        ray_use = np.argmin(np.abs(ray_angles - ac_roll))  #the index of the ray whose angle is closest to that of ac_roll; i.e., the "pseudo-nadir" ray
                        
                        if abs(ac_roll) >= 10:
                            apr_profile_roll10 += 1
                        
                        prof_height = alt3d[time_idx,ray_use,:].values
                        prof_dbz = ku_data[time_idx,ray_use,:].values
                        
                        #grab the height/dbz data above 1.5km
                        idx_1500m = np.argmin(np.abs(prof_height - 1500))
                        prof_height_above1500m = np.flip(prof_height[:idx_1500m + 1])  #height profile above 1.5km, in ascending order
                        prof_dbz_above1500m = np.flip(prof_dbz[:idx_1500m + 1])  #Ku-band profile above 1.5km, in ascending order
                        
                        #store all the profile's height/dbz data in long 1-D concatenated arrays
                        if new_ku_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                            height_concat = prof_height.copy()
                            dbz_concat = prof_dbz.copy()
                            
                            #create a DataFrame of values above 1.5km, in order to calculate median/quartile reflectivity profile for the given case
                                #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                    #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile     
                            height_profiles_above1500m_df = pd.DataFrame(prof_height_above1500m)  #height profile from 1500m to top of profile
                            dbz_profiles_above1500m_df = pd.DataFrame(prof_dbz_above1500m)        #dbz profile from 1500m to top of profile
                            new_ku_array = False
                        else:
                            height_concat = np.concatenate((height_concat, prof_height))
                            dbz_concat = np.concatenate((dbz_concat, prof_dbz))
                            
                            #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
                            height_profiles_above1500m_df = pd.concat((height_profiles_above1500m_df, pd.Series(prof_height_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                            dbz_profiles_above1500m_df = pd.concat((dbz_profiles_above1500m_df, pd.Series(prof_dbz_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                        
                        # if vel_good:
                        #     prof_vel = vel_data[time_idx,ray_use,:].values
                            
                        #     #store all the profile's velocity data in long 1-D concatenated arrays if 
                        #     #the profile has Ku data > 0 dBZ above 1.5km (i.e. omits clear profiles)
                            
                        #     if np.nanmax(prof_dbz_above1500m) > 0:
                        #         if new_vel_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                        #             vel_concat = prof_vel.copy()
                        #             height_vel_concat = prof_height.copy()
                        #             new_vel_array = False
                        #         else:
                        #             vel_concat = np.concatenate((vel_concat, prof_vel))
                        #             height_vel_concat = np.concatenate((height_vel_concat, prof_height))
                        #     else:  #clear profiles (which may have noisy velocity data) are omitted from the velocity CFAD
                        #         pass
                        # else:
                        #     pass
                            
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                            
                            
                else:
                    first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                    
                apr_file.close()
                               
            elif desired_date[:4] == '2019':    #CAMP2Ex
            
                ku_band = 'lores_zhh14' #Ku-band reflectivity, (Scan, Ray, Range Bin)
                vel = 'lores_vel14c'    #Mean Doppler Velocity at Ku-band CORRECTED by surface-reference technique (surface Doppler velocity is subtracted and free of aliasing), (Scan, Ray, Range Bin)
            
                apr_file = xr.open_dataset(os.path.join(apr_folder, apr_filepath))
                file_start_date = apr_filepath[35:43]  #this date determines how the 'Time' variable is calculated (will be seconds since midnight UTC of this date (YYYMMMDD))
                
                if ku_band in apr_file.keys():
                    
                    try:   #some APR files, at least in the preliminary data, have corrupted Ku-band data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        ku_data = apr_file[ku_band][:]
                    except:
                        first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                        apr_file.close()
                        continue  #both Ku-band and velocity CFADs rely on Ku-band data availability
    
                    try:   #some APR files, at least in the preliminary data, have corrupted velocity data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        vel_data = apr_file[vel][:]
                        vel_good = True
                    except:
                        vel_good = False                    
                
                    #grab the radar variables of interest
                    time = apr_file['time'][:]          #For 'lores': Time of scan, in seconds since midnight UTC of file_start_date (YYYMMMDD)
                    alt3d = apr_file['lores_alt3D'][:]
                    roll = apr_file['lores_roll'][:]
                   
                    if apr_filepath == apr_files_use[first_good_Ku_file_index] or apr_filepath == apr_files_use[-1]:
                    #the complete APR file time range may not need to be used, so need to locate the closest time (and corresponding index) to the desired start/end time
                        
                        #Convert APR times to datetimes
                        time_dates = np.empty(time.shape, dtype=object)
                        for i in np.arange(0, time.shape[0]):
                            #hour, second automatically revert to midnight (hour = 0, seconds = 0) for '%Y%m%d'
                            time_dates[i] = datetime.strptime(file_start_date, '%Y%m%d') + timedelta(seconds = float(time[i].values))
                        
                        unique_apr_times = time_dates[:]  #all the times in the given APR file
                     
                        if len(apr_files_use) == 1:  #i.e. apr_filepath == apr_files_use[first_good_Ku_file_index] and apr_filepath == apr_files_use[-1]
                        
                            #find the closest time (and its corresponding index) to the desired start_time
                            if key == 2 or key == '17b':
                                desired_start_time = datetime.strptime(desired_date_next + start_time, '%Y%m%d%H%M%S')
                            else:
                                desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                        
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date_next + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                                    
                        elif apr_filepath == apr_files_use[first_good_Ku_file_index]:  #i.e. the first APR file in the apr_files_use list
                            end_time_idx = time.shape[0] - 1
                            
                            #find the closest time (and its corresponding index) to the desired start_time
                            if key == 2 or key == '17b':
                                desired_start_time = datetime.strptime(desired_date_next + start_time, '%Y%m%d%H%M%S')
                            else:
                                desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                                    
                        else:  #apr_filepath == apr_files_use[-1]  #i.e. the last APR file in the apr_files_use list
                            start_time_idx = 0
                            
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date_next + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                            
                    else:  #the entire APR file is within the desired time range, so set the start/end indices to the first/last indices of the APR file
                        start_time_idx = 0
                        end_time_idx = time.shape[0] - 1
                            
                    #numpy.histogram2d(dBZ, z) (Zagrodnik et al., 2019) CFAD method:
                    
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                       
                    
                    #loop through the given APR file’s valid times/scans/profiles
                    for time_idx in range(start_time_idx, end_time_idx + 1):          #time_idx represents the scan number
                        
                        profile_time = datetime.strptime(file_start_date, '%Y%m%d') + timedelta(seconds = float(time[time_idx].values))
                        ecco_ip = np.argmin(abs(ecco_df['Full_Datetime'] - profile_time))  #abs() necessary to properly subtract datetime objects
                                                                                            #returns the integer position of the minimum value
                        #print (ecco_df['Full_Datetime'].iloc[ecco_index])           #to check that the correct time was chosen
                        classification = ecco_df['Classification'].iloc[ecco_ip]     #grab the classification of the ECCO-V time closest to the given profile time (may/should be the exact time)
                        
                        if classification not in ['0', '1', '2']:
                            raise ValueError(f'{classification} is not a valid ECCO-V classification.')
                    
                        #skip profiles with an ECCO-V classification of "no classification" (0) or "stratiform" (1)
                        #(i.e., only use "convective" (2) profiles)
                        if classification == '0' or classification == '1':
                            continue
                        
                        # #skip profiles with an ECCO-V classification of "no classification" (0) or "convective" (2)
                        # #(i.e., only use "stratiform" (1) profiles)
                        # if classification == '0' or classification == '2':
                        #     continue
                        
                        total_apr_profiles += 1
                        
                        #choose the "nadir" ray factoring in aircraft roll
                        ac_roll = np.nanmean(roll[time_idx,:])  #roll varies slightly w/ray, so take the average roll value for a given scan and use that for ray adjustment
                        ray_use = np.argmin(np.abs(ray_angles - ac_roll))  #the index of the ray whose angle is closest to that of ac_roll; i.e., the "pseudo-nadir" ray
                        
                        if abs(ac_roll) >= 10:
                            apr_profile_roll10 += 1
                        
                        prof_height = alt3d[time_idx,ray_use,:].values
                        prof_dbz = ku_data[time_idx,ray_use,:].values
                        
                        #grab the height/dbz data above 1.5km
                        idx_1500m = np.argmin(np.abs(prof_height - 1500))
                        prof_height_above1500m = np.flip(prof_height[:idx_1500m + 1])  #height profile above 1.5km, in ascending order
                        prof_dbz_above1500m = np.flip(prof_dbz[:idx_1500m + 1])  #Ku-band profile above 1.5km, in ascending order
                        
                        #store all the profile's height/dbz data in long 1-D concatenated arrays
                        if new_ku_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                            height_concat = prof_height.copy()
                            dbz_concat = prof_dbz.copy()
                            
                            #create a DataFrame of values above 1.5km, in order to calculate median/quartile reflectivity profile for the given case
                                #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                    #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile     
                            height_profiles_above1500m_df = pd.DataFrame(prof_height_above1500m)  #height profile from 1500m to top of profile
                            dbz_profiles_above1500m_df = pd.DataFrame(prof_dbz_above1500m)        #dbz profile from 1500m to top of profile
                            new_ku_array = False
                        else:
                            height_concat = np.concatenate((height_concat, prof_height))
                            dbz_concat = np.concatenate((dbz_concat, prof_dbz))
                            
                            #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
                            height_profiles_above1500m_df = pd.concat((height_profiles_above1500m_df, pd.Series(prof_height_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                            dbz_profiles_above1500m_df = pd.concat((dbz_profiles_above1500m_df, pd.Series(prof_dbz_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                        
                        # if vel_good:
                        #     prof_vel = vel_data[time_idx,ray_use,:].values
                            
                        #     #store all the profile's velocity data in long 1-D concatenated arrays if 
                        #     #the profile has Ku data > 0 dBZ above 1.5km (i.e. omits clear profiles)
                            
                        #     if np.nanmax(prof_dbz_above1500m) > 0:
                        #         if new_vel_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                        #             vel_concat = prof_vel.copy()
                        #             height_vel_concat = prof_height.copy()
                        #             new_vel_array = False
                        #         else:
                        #             vel_concat = np.concatenate((vel_concat, prof_vel))
                        #             height_vel_concat = np.concatenate((height_vel_concat, prof_height))
                        #     else:  #clear profiles (which may have noisy velocity data) are omitted from the velocity CFAD
                        #         pass
                        # else:
                        #     pass
                            
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                            
                            
                else:
                    first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                    
                apr_file.close()
                
            else:
                sys.exit('Not a CPEX or CPEX-AW or CPEX-CV or CAMP2Ex case')
                
        print ('Case {} complete'.format(key))
       
    #calculate median/quantile dBZ profiles for the given case
    
    #replace blank data (-99.99) with NaNs
    dbz_profiles_above1500m_df[dbz_profiles_above1500m_df <= -99] = np.nan
    height_profiles_above1500m_df[height_profiles_above1500m_df < 0] = np.nan
    
    #convert all the dropsonde's Ku-band data from dBZ to mm^6 m^-3
    Z_df = 10**(dbz_profiles_above1500m_df / 10)
    
    #calculate the dropsonde's median/quantile heights and reflectivity (in mm^6 m^-3) and standard deviation at each height level
    #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
        #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
        
    median_height_profile_above1500m = height_profiles_above1500m_df.median(axis = 1)  #axis of 1 = across all columns (so for each row); NaNs are ignored by default in Pandas
    Z_median = Z_df.median(axis = 1)  #axis of 1 = across all columns (so for each row); NaNs are ignored by default in Pandas
    Z_Q1 = Z_df.quantile(q = 0.25, axis = 1)
    Z_Q3 = Z_df.quantile(q = 0.75, axis = 1)
    #Z_std = Z_df.std(axis = 1)
    
    #convert the dropsonde's median/quartile reflectivity and standard deviation profile back to dBZ
    medianDBZ_profile_above1500m = 10 * np.log10(Z_median)
    Q1DBZ_profile_above1500m = 10 * np.log10(Z_Q1)
    Q3DBZ_profile_above1500m = 10 * np.log10(Z_Q3)
    #stdDBZ_profile_above1500m = 10 * np.log10(Z_std)      
        
    
    #setting the height, reflectivity, and velocity bin edges, along with their associated meshgrids
    height_edges = height_bin_edges
    dbz_edges = dbz_bin_edges
    #dbz_meshgrid, height_meshgrid = np.meshgrid(dbz_edges, height_edges)
    
    vel_edges = vel_bin_edges
    #vel_meshgrid, height_vel_meshgrid = np.meshgrid(vel_edges, height_edges)
    
    #setting the height, reflectivity, and velocity bin centers and their associated meshgrids for contourf plotting
    height_centers = (height_edges[:-1] + height_edges[1:]) / 2    
    dbz_centers = (dbz_edges[:-1] + dbz_edges[1:]) / 2
    dbz_centers_meshgrid, height_centers_meshgrid = np.meshgrid(dbz_centers, height_centers)
    
    vel_centers = (vel_edges[:-1] + vel_edges[1:]) / 2
    vel_centers_meshgrid, height_vel_centers_meshgrid = np.meshgrid(vel_centers, height_centers)
  
    #create the 2D histogram of values/frequencies for Ku-band data
    cfad_array, xedges, yedges = np.histogram2d(dbz_concat, height_concat, bins = (dbz_edges,height_edges))
    
    #transpose cfad_array shape (rows, columns) to be (height, dbz) instead of (dbz, height)
    cfad_array = cfad_array.T
    #cfad_array = np.log10(cfad_array)  #creates log-weighted CFADs
    contours = np.linspace(0, np.nanmax(cfad_array), 21)
    colorbar_label = 'Total Frequency [#]'
    var_label = 'Ku-band Reflectivity [dBZ]'
    save_label = 'CFAD_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Ku.png'
    plot_CFAD(cfad_array, contours, colorbar_label, save_label, var_label, dbz_centers_meshgrid, height_centers_meshgrid, dbz_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = True)
    
    #normalize the CFAD and make a normalized CFAD plot if that is also desired
    if normalize:
        #cfad_array = cfad_array / np.nanmax(cfad_array) * 100
        height_bin_maxs = np.nanmax(cfad_array, axis = 1)   #normalize the CFAD by max count in each height bin
        cfad_array = cfad_array / height_bin_maxs[:, np.newaxis] * 100
        contours = np.arange(0,101,5)
        colorbar_label = 'Normalized Frequency [%]'
        save_label = 'CFADnorm_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Ku.png'
        plot_CFAD(cfad_array, contours, colorbar_label, save_label, var_label, dbz_centers_meshgrid, height_centers_meshgrid, dbz_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = True)
        
        
    # #create the 2D histogram of values/frequencies for Doppler Velocity data
    # cfad_array_vel, xedges, yedges = np.histogram2d(vel_concat, height_vel_concat, bins = (vel_edges,height_edges))
    
    # #transpose cfad_array shape (rows, columns) to be (height, vel) instead of (vel, height)
    # cfad_array_vel = cfad_array_vel.T
    # #cfad_array_vel = np.log10(cfad_array_vel)  #creates log-weighted CFADs
    # contours = np.linspace(0, np.nanmax(cfad_array_vel), 21)
    # colorbar_label = 'Total Frequency [#]'
    # var_label = 'Mean Doppler Velocity [m/s]'
    # save_label = 'CFAD_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Vel.png'
    # plot_CFAD(cfad_array_vel, contours, colorbar_label, save_label, var_label, vel_centers_meshgrid, height_vel_centers_meshgrid, vel_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = False)
    
    # #normalize the CFAD and make a normalized CFAD plot if that is also desired
    # if normalize:
    #     #cfad_array_vel = cfad_array_vel / np.nanmax(cfad_array_vel) * 100
    #     height_bin_maxs = np.nanmax(cfad_array_vel, axis = 1)   #normalize the CFAD by max count in each height bin
    #     cfad_array_vel = cfad_array_vel / height_bin_maxs[:, np.newaxis] * 100
    #     contours = np.arange(0,101,5)
    #     colorbar_label = 'Normalized Frequency [%]'
    #     save_label = 'CFADnorm_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Vel.png'
    #     plot_CFAD(cfad_array_vel, contours, colorbar_label, save_label, var_label, vel_centers_meshgrid, height_vel_centers_meshgrid, vel_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = False)
        
    # #print ('Percent of profiles with A/C roll >= 10 degrees:', apr_profile_roll10 / total_apr_profiles * 100)
    
    # return cfad_array, cfad_array_vel, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m
    return cfad_array, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m

def plot_CFAD(cfad_array, contours, colorbar_label, save_label, var_label, var_centers_meshgrid, height_centers_meshgrid, var_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = True):
    
    """Plot a contourf CFAD given a CFAD 2-D array, contour levels, plot/image labels, and variable/height meshgrids"""
    
    #plot the CFAD
    fig, ax = plt.subplots(1,1, figsize=(21,21))
    cmap = mplc.ListedColormap(['#ffffff', '#d8fcfa', '#bef5f6', '#aaedf1', '#98e4ec', '#89dae7', '#7cd0e2', 
                                '#70c6dd', '#65bcd9', '#5cb2d4', '#53a8cf', '#4b9dca', '#4393c5', '#3c89c0', 
                                '#357ebb', '#2f74b6', '#286ab1', '#2160ac', '#1956a7', '#0f4ca2', '#00429d'])
    
    #the cmap below omits the white fill at the beginnning
    # cmap = mplc.ListedColormap(['#d8fcfa', '#bef5f6', '#aaedf1', '#98e4ec', '#89dae7', '#7cd0e2', 
    #                             '#70c6dd', '#65bcd9', '#5cb2d4', '#53a8cf', '#4b9dca', '#4393c5', '#3c89c0', 
    #                             '#357ebb', '#2f74b6', '#286ab1', '#2160ac', '#1956a7', '#0f4ca2', '#00429d'])
    
    #cs = ax.pcolormesh(dbz_meshgrid, height_meshgrid, cfad_array, cmap = cmap)
    #cs = ax.contour(dbz_centers_meshgrid, height_centers_meshgrid, cfad_array, levels = contours, cmap = cmap, linewidths = 1)
    cs = ax.contourf(var_centers_meshgrid, height_centers_meshgrid / 1000, cfad_array, levels = contours, cmap = cmap)
    if dbz_plot:
        ax.plot(medianDBZ_profile_above1500m, median_height_profile_above1500m / 1000, color = 'k', linestyle = '-', linewidth = 5, label = 'Case ' + ','.join(map(str, list(case_dict.keys()))) + ' Median Profile') 
        ax.plot(Q1DBZ_profile_above1500m, median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5, label = 'Case ' + ','.join(map(str, list(case_dict.keys()))) + ' Q1 and Q3 Profiles')
        ax.plot(Q3DBZ_profile_above1500m, median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5)
        ax.legend(loc = 'upper right')
    ax.set_ylabel('Altitude [m]', fontsize=30, fontweight = 'bold')
    ax.set_xlabel(var_label, fontsize=30, fontweight = 'bold')
    ax.tick_params(length = 15, width = 5, labelsize = 25)
    #ax.set_title('Total CFAD for Case ' + ','.join(map(str, list(case_dict.keys()))), fontsize=35, fontweight = 'bold')
    ax.set_title('Case ' + ','.join(map(str, list(case_dict.keys()))) + ' Normalized CFAD (Only Convective Regions)', fontsize=35, fontweight = 'bold')
    ax.set_ylim([(height_edges[0] + 250) / 1000, (height_edges[-1] - 250) / 1000])
    #ax.set_xlim([var_edges[0],var_edges[-1]])
    ax.set_xlim([5,60])  #for Ku-band
    
    #set the colorbar axis
    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.08,
                       ax.get_position().x1-ax.get_position().x0, 0.02])    #Left, bottom, width, height (all [0,1])

    #create the colorbar
    cbar = plt.colorbar(cs, cax = cax, orientation = 'horizontal')
    cbar.ax.tick_params(length = 10, width = 3, labelsize = 23)
    cbar.set_label(label = colorbar_label, fontsize = 30, fontweight = 'bold')
    
    #save the figure
    plt.savefig(''.join(['/Users/ben/Desktop/', save_label]), bbox_inches = 'tight')
    plt.close()


def difference_CFAD(case_dict_1, case_dict_2, height_bin_edges, dbz_bin_edges, vel_bin_edges, diff_plot_title, save_label_manual):

    """"Calculate and plot Ku-band and Doppler Velocity difference (normalized) CFAD 2-D arrays for 
        the given pair of dictionaries containing given cases and their respective time ranges
    
    Parameters
    ----------
    case_dict_1:  dictionary of cases for which to calculate the first normalized CFAD; 
                  keys should be case numbers;
                  values should be 3-element lists of case date and start/end times, 
                  with date strings formatted as YYYYMMDD and time strings formatted as HHMMSS
                  
    case_dict_2:  dictionary of cases for which to calculate the second normalized CFAD; 
                  keys should be case numbers;
                  values should be 3-element lists of case date and start/end times, 
                  with date strings formatted as YYYYMMDD and time strings formatted as HHMMSS
    
    height_bin_edges:  a 1-D array of height [m] bin edges used to create the 2-D difference CFAD array/histogram    

    dbz_bin_edges:  a 1-D array of reflectivity [dBZ] bin edges used to create the 2-D difference CFAD array/histogram 

    vel_bin_edges:  a 1-D array of Doppler velocity [m/s] bin edges used to create the 2-D difference CFAD array/histogram
                        
    return:  2-D (normalized) difference CFAD plots (case_dict_2 - case_dict_1)"""

    #calculate the Ku-band and Doppler Velocity 2-D normalized CFADs for each case
    # first_dbz_CFAD, first_vel_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m = CFAD(case_dict_1, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    # second_dbz_CFAD, second_vel_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m = CFAD(case_dict_2, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    first_dbz_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m = CFAD(case_dict_1, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    second_dbz_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m = CFAD(case_dict_2, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    
    #calculate the Ku-band and Doppler Velocity 2-D difference CFADs
    dbz_diff_CFAD = second_dbz_CFAD - first_dbz_CFAD
    # vel_diff_CFAD = second_vel_CFAD - first_vel_CFAD
    
    #calculate the maximum magnitude for each CFAD and create contours for diverging colormaps accordingly
    
    #dbz_highest_mag = np.nanmax(np.abs(dbz_diff_CFAD))  #creates unique colorbar range for each difference CFAD plot
    dbz_highest_mag = 100  #creates uniform colorbar range across all difference CFAD plots
    dbz_contours = np.linspace(-dbz_highest_mag, dbz_highest_mag, 41)  #an odd number of intervals (21) guarantees a contour at 0 for a range from -x to x
    
    # #vel_highest_mag = np.nanmax(np.abs(vel_diff_CFAD))  #creates unique colorbar range for each difference CFAD plot
    # vel_highest_mag = 100  #creates uniform colorbar range across all difference CFAD plots
    # vel_contours = np.linspace(-vel_highest_mag, vel_highest_mag, 41)  #an odd number of intervals (21) guarantees a contour at 0 for a range from -x to x
    
    #grab the case numbers of the 2 cases/case sets being differenced and create appropriate image names
    case1_num = ','.join(map(str, list(case_dict_1.keys())))
    case2_num = ','.join(map(str, list(case_dict_2.keys())))
    cases = case1_num + 'and' + case2_num
    colorbar_label = 'Differential Normalized Frequency [%]'
    #dbz_save_label = 'diff_CFADnorm_Cases_' + cases + '_Ku.png'
    dbz_save_label = save_label_manual + '_Ku.png'
    #vel_save_label = 'diff_CFADnorm_Cases_' + cases + '_Vel.png'
    vel_save_label = save_label_manual + '_Vel.png'

    #plot the Ku-band and Doppler Velocity 2-D difference CFADs
    plot_diffCFAD(dbz_diff_CFAD, dbz_contours, colorbar_label, dbz_save_label, 'Ku-band Reflectivity [dBZ]', dbz_centers_meshgrid, height_centers_meshgrid, dbz_bin_edges, height_bin_edges, case1_num, case2_num, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m, diff_plot_title, dbz_plot = False) 
    # plot_diffCFAD(vel_diff_CFAD, vel_contours, colorbar_label, vel_save_label, 'Mean Doppler Velocity [m/s]', vel_centers_meshgrid, height_vel_centers_meshgrid, vel_bin_edges, height_bin_edges, case1_num, case2_num, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m, dbz_plot = False)
  

def plot_diffCFAD(cfad_array, contours, colorbar_label, save_label, var_label, var_centers_meshgrid, height_centers_meshgrid, var_edges, height_edges, case1_num, case2_num, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m, diff_plot_title, dbz_plot = True):
    
    """Plot a contourf difference CFAD given a difference CFAD 2-D array, contour levels, plot/image labels, and variable/height meshgrids"""
    
    #plot the CFAD
    fig, ax = plt.subplots(1,1, figsize=(21,21))
    # cmap = mplc.ListedColormap(['#00429d', '#2855a6', '#3e68af', '#507bb8', '#618fc1', '#73a3ca', '#85b8d3', 
    #                             '#99ccdc', '#b0e0e6', '#cef2f1', '#ffffff', '#ffe5e7', '#ffcbcf', '#ffafb7', 
    #                             '#ff929e', '#f57789', '#e95d76', '#d94364', '#c62a54', '#af1046', '#93003a'])
    
    cmap = mplc.ListedColormap(['#00429d', '#074ca2', '#0e57a8', '#1561ad', '#1d6bb2', '#2475b8', '#2b7fbd', 
                                '#3289c2', '#3994c8', '#409ecd', '#48a8d2', '#4fb3d8', '#56bddd', '#5ec8e3', 
                                '#65d3e8', '#6dddee', '#74e8f4', '#7cf3f9', '#83feff', '#cbffff', '#ffffff', 
                                '#fef3f0', '#fce7e0', '#fbdad0', '#facec1', '#f8c2b1', '#f7b5a1', '#f5a890', 
                                '#f49b7f', '#f28d6e', '#f17f5c', '#ef6f48', '#ed5f33', '#eb4c1b', '#e13f18', 
                                '#d5351e', '#c82b23', '#bc2128', '#af162e', '#a10b34', '#93003a'])
    
    #cs = ax.pcolormesh(dbz_meshgrid, height_meshgrid, cfad_array, cmap = cmap)
    #cs = ax.contour(dbz_centers_meshgrid, height_centers_meshgrid, cfad_array, levels = contours, cmap = cmap, linewidths = 1)
    cs = ax.contourf(var_centers_meshgrid, height_centers_meshgrid / 1000, cfad_array, levels = contours, cmap = cmap)
    if dbz_plot:
        ax.plot(second_medianDBZ_profile_above1500m, second_median_height_profile_above1500m / 1000, color = 'k', linestyle = '-', linewidth = 5, label = 'Case ' + case2_num + ' Median Profile') 
        ax.plot(second_Q1DBZ_profile_above1500m, second_median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5, label = 'Case ' + case2_num + ' Q1 and Q3 Profiles')
        ax.plot(second_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5)
        
        ax.plot(first_medianDBZ_profile_above1500m, first_median_height_profile_above1500m / 1000, color = 'darkgoldenrod', linestyle = '-', linewidth = 5, label = 'Case ' + case1_num + ' Median Profile') 
        ax.plot(first_Q1DBZ_profile_above1500m, first_median_height_profile_above1500m / 1000, color = 'darkgoldenrod', linestyle = '--', linewidth = 5, label = 'Case ' + case1_num + ' Q1 and Q3 Profiles')
        ax.plot(first_Q3DBZ_profile_above1500m, first_median_height_profile_above1500m / 1000, color = 'darkgoldenrod', linestyle = '--', linewidth = 5)
        
        ax.legend(loc = 'upper right')
        
    ax.set_ylabel('Altitude [km]', fontsize=30, fontweight = 'bold', labelpad = 20.0)
    ax.set_xlabel(var_label, fontsize=30, fontweight = 'bold', labelpad = 10.0)
    ax.tick_params(length = 15, width = 5, labelsize = 25)
    ax.set_title(diff_plot_title, fontsize=35, fontweight = 'bold')
    ax.set_ylim([(height_edges[0] + 250) / 1000, (height_edges[-1] - 250) / 1000])
    #ax.set_ylim([(height_edges[0] + 250) / 1000, 5.250])    #for CAMP2Ex Case 10 vs. Case 1
    #ax.set_ylim([(height_edges[0] + 250) / 1000, 4.750])   #for CAMP2Ex Case 17 vs. Case 15
    #ax.set_xlim([var_edges[0],var_edges[-1]])
    ax.set_xlim([5,60])  #for Ku-band
    
    #set the colorbar axis
    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.08,
                       ax.get_position().x1-ax.get_position().x0, 0.02])    #Left, bottom, width, height (all [0,1])

    #create the colorbar
    cbar = plt.colorbar(cs, cax = cax, orientation = 'horizontal')
    cbar.ax.tick_params(length = 10, width = 3, labelsize = 23)
    cbar.set_label(label = colorbar_label, fontsize = 30, fontweight = 'bold')
    
    #save the figure
    #plt.savefig(''.join(['/Users/ben/Desktop/CPEX-CV/Coding/CFAD_plots/Case', case1_num, 'and', case2_num, '_Comparison/', save_label]), bbox_inches = 'tight')
    plt.savefig(''.join(['/Users/ben/Desktop/', save_label]), bbox_inches = 'tight')    
    plt.close()


#run the CFAD function to plot the CFADs

#######################################################################################################
#composite Isolated and Organized convective intensity interregional comparisons for tropical West Atlantic (CPEX, CPEX-AW), East Atlantic (CPEX-CV), and Northwest Pacific (CAMP2Ex)

#only convective regions (all lifecycle stages)
difference_CFAD(eatl_isolated_case_dict, watl_isolated_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Isolated, Only Convective Regions)', 'diffCFAD_WATL_minus_EATL_Isolated_all_lifecycles_Conv')           #WATL minus EATL
difference_CFAD(nwpac_isolated_case_dict, watl_isolated_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Isolated, Only Convective Regions)', 'diffCFAD_WATL_minus_NWPAC_Isolated_all_lifecycles_Conv')        #WATL minus NWPAC
difference_CFAD(nwpac_isolated_case_dict, eatl_isolated_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Isolated, Only Convective Regions)', 'diffCFAD_EATL_minus_NWPAC_Isolated_all_lifecycles_Conv')        #EATL minus NWPAC
difference_CFAD(eatl_organized_case_dict, watl_organized_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Organized, Only Convective Regions)', 'diffCFAD_WATL_minus_EATL_Organized_all_lifecycles_Conv')       #WATL minus EATL
difference_CFAD(nwpac_organized_case_dict, watl_organized_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Organized, Only Convective Regions)', 'diffCFAD_WATL_minus_NWPAC_Organized_all_lifecycles_Conv')    #WATL minus NWPAC
difference_CFAD(nwpac_organized_case_dict, eatl_organized_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Organized, Only Convective Regions)', 'diffCFAD_EATL_minus_NWPAC_Organized_all_lifecycles_Conv')    #EATL minus NWPAC

#only convective regions (only if case was at least partly collected during mature lifecycle stage)
difference_CFAD(eatl_isolated_case_dict_growing_or_mature, watl_isolated_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Isolated, Only Convective Regions)', 'diffCFAD_WATL_minus_EATL_Isolated_partially_growing_or_mature_lifecycles_Conv')           #WATL minus EATL
difference_CFAD(nwpac_isolated_case_dict_growing_or_mature, watl_isolated_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Isolated, Only Convective Regions)', 'diffCFAD_WATL_minus_NWPAC_Isolated_partially_growing_or_mature_lifecycles_Conv')        #WATL minus NWPAC
difference_CFAD(nwpac_isolated_case_dict_growing_or_mature, eatl_isolated_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Isolated, Only Convective Regions)', 'diffCFAD_EATL_minus_NWPAC_Isolated_partially_growing_or_mature_lifecycles_Conv')        #EATL minus NWPAC
difference_CFAD(eatl_organized_case_dict_growing_or_mature, watl_organized_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Organized, Only Convective Regions)', 'diffCFAD_WATL_minus_EATL_Organized_partially_growing_or_mature_lifecycles_Conv')       #WATL minus EATL
difference_CFAD(nwpac_organized_case_dict_growing_or_mature, watl_organized_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Organized, Only Convective Regions)', 'diffCFAD_WATL_minus_NWPAC_Organized_partially_growing_or_mature_lifecycles_Conv')    #WATL minus NWPAC
difference_CFAD(nwpac_organized_case_dict_growing_or_mature, eatl_organized_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Organized, Only Convective Regions)', 'diffCFAD_EATL_minus_NWPAC_Organized_partially_growing_or_mature_lifecycles_Conv')    #EATL minus NWPAC



### Stratiform Regions Only

In [ ]:
#CFADs (with and without convective-stratiform partitioning)
#adapted from CFAD_AGUpapers_only_pseudonadir_ray_and_ECCO.py

matplotlib.rcParams['font.family'] = 'arial'
# matplotlib.rcParams['axes.labelsize'] = 14
# matplotlib.rcParams['axes.titlesize'] = 14
# matplotlib.rcParams['xtick.labelsize'] = 12
# matplotlib.rcParams['ytick.labelsize'] = 12
matplotlib.rcParams['legend.fontsize'] = 24
#matplotlib.rcParams['legend.facecolor'] = 'w'

#create the dictionary of cases for which you want to plot a total CFAD and create the height/reflectivity bin edges

# #CAMP2Ex cases (Cases 1-14 are Isolated; Cases 15-19 are Organized)
# case1_dict = {1: ['20190829', '234500','040000']}      #Good APR-3 data coverage
# case2_dict = {2: ['20190829', '041500','051000']}      #Meh APR-3 data coverage, probably don’t use for CFADs
# case3_dict = {3: ['20190831', '000000', '001000']}     #No APR-3 data coverage, techincally case starts at 23:50 UTC on 20190830, but no APR-3 data files until 20190831
# case4_dict = {4: ['20190831', '050000', '055000']}     #Meh APR-3 data coverage, probably don’t use for CFADs
# case5_dict = {5: ['20190904', '014500', '050000' ]}    #No APR-3 data coverage
# case6_dict = {6: ['20190904', '050000', '060500']}     #No APR-3 data coverage
# case7_dict = {7: ['20190904', '060500', '071500']}     #No APR-3 data coverage
# case8_dict = {8: ['20190907', '062000', '071000']}     #Good APR-3 data coverage
# case9_dict = {'9a': ['20190909', '003500', '005500'], '9b': ['20190909', '023000', '024500']}   #No APR-3 data coverage
# case10_dict = {10: ['20190909', '005500', '022500']}   #Good APR-3 data coverage
# case11_dict = {11: ['20191001', '220000', '224500']}   #No APR-3 data coverage
# case12_dict = {12: ['20191002', '053000', '061500']}   #No APR-3 data coverage
# case13_dict = {13: ['20191003', '231000', '004500']}   #No APR-3 data coverage
# case14_dict = {14: ['20191004', '004500', '010500']}   #No APR-3 data coverage
# case15_dict = {'15a': ['20190907', '002500', '024000'], '15b': ['20190907', '031000', '040500']}   #Good APR-3 data coverage and it is through the center of the storm (though not at its most intense)
# case16_dict = {'16a': ['20190907', '024500', '031000'], '16b': ['20190907', '040500', '044500'], '16c': ['20190907', '071000', '074000']}   #Little APR-3 data coverage (definitely not good enough for CFADs), even though the flight goes right through the growing storm; APR-3 reflectivity from 02:45:00 – 03:10:00 UTC is from the infancy of the storm (and maybe not the storm at all) and so is not really representative of the storm
# case17_dict = {'17a': ['20190915', '223500', '230500'], '17b': ['20190915', '031000', '044500']}   #Good APR-3 data coverage and it is through the center of the storm
# case18_dict = {18: ['20190917', '014500', '061500']}   #Meh good APR-3 data coverage, but coverage really only on the outer edge of the convection (nowhere close to the center/strongest parts of the storm)
# case19_dict = {19: ['20191005', '023000', '065000']}   #Bad APR-3 data coverage, and coverage only on the far outer edge of the convection after 6 UTC (nowhere close to center/strongest parts of the storm)

nwpac_isolated_case_dict = {1: ['20190829', '234500','040000'], 8: ['20190907', '062000', '071000'], 10: ['20190909', '005500', '022500']}   #cases 1, 8, 10 above
nwpac_organized_case_dict = {'15a': ['20190907', '002500', '024000'], '15b': ['20190907', '031000', '040500'], 
                             '17a': ['20190915', '223500', '230500'], '17b': ['20190915', '031000', '044500']}   #cases 15, 17 above

#cases from nwpac_xxx_case_dict above that were at least partly collected during growing or mature lifecycle stages
nwpac_isolated_case_dict_growing_or_mature = {1: ['20190829', '234500','040000'], 10: ['20190909', '005500', '022500']}   #cases 1, 10 above
nwpac_organized_case_dict_growing_or_mature = {'15a': ['20190907', '002500', '024000'], '15b': ['20190907', '031000', '040500']}   #case 15 above

# #CPEX(-AW) cases
# case1_dict = {1: ['20170610','194655','221900']}
# case2_dict = {2: ['20170624','180000','194800']}
# case3_dict = {3: ['20170624','201200','220000']}
# case4_dict = {4: ['20170615', '184840', '205000']}
# case5_dict = {5: ['20170616', '182451', '220600']}
# case6_dict = {6: ['20170601', '175849', '220700']}
# case7_dict = {7: ['20170606', '185211', '215000']}
# case8_dict = {8: ['20170617', '184650', '220000']}
# case13_dict = {13: ['20170611', '180100', '203400']}
# case14_dict = {14: ['20210821', '221800', '234145']}
# case16_dict = {16: ['20210824', '181545', '195745']}

watl_isolated_case_dict = {1: ['20170610','194655','221900'], 2: ['20170624','180000','194800'], 3: ['20170624','201200','220000']}   #cases 1, 2, 3 above
watl_organized_case_dict = {4: ['20170615', '184840', '205000'], 5: ['20170616', '182451', '220600'], 6: ['20170601', '175849', '220700'],
                            7: ['20170606', '185211', '215000'], 8: ['20170617', '184650', '220000'], 13: ['20170611', '180100', '203400'],
                            14: ['20210821', '221800', '234145'], 16: ['20210824', '181545', '195745']}   #cases 4, 5, 6, 7, 8, 13, 14, 16 above

#cases from watl_xxx_case_dict above that were at least partly collected during growing or mature lifecycle stages (turns out to be all the cases from above)
watl_isolated_case_dict_growing_or_mature = {1: ['20170610','194655','221900'], 2: ['20170624','180000','194800'], 3: ['20170624','201200','220000']}   #cases 1, 2, 3 above
watl_organized_case_dict_growing_or_mature = {4: ['20170615', '184840', '205000'], 5: ['20170616', '182451', '220600'], 6: ['20170601', '175849', '220700'],
                                              7: ['20170606', '185211', '215000'], 8: ['20170617', '184650', '220000'], 13: ['20170611', '180100', '203400'],
                                              14: ['20210821', '221800', '234145'], 16: ['20210824', '181545', '195745']}   #cases 4, 5, 6, 7, 8, 13, 14, 16 above

# #CPEX-CV cases (Cases 1-7 are Isolated; Cases 8-22 are Organized (ignore Case 19 (TC)); Cases 23-24 are Scattered)
# case1_dict = {1: ['20220909','161000','171000']}
# case2_dict = {2: ['20220909','173500','191000']}
# case3_dict = {3: ['20220910','190500','194200']}
# case4_dict = {4: ['20220910', '202500', '204500']}
# case5_dict = {5: ['20220916', '155500', '163500']}
# case6_dict = {6: ['20220920', '071000', '073500']}
# case7_dict = {7: ['20220920', '083000', '090000']}
# case8_dict = {8: ['20220906', '110000', '120000']}
# case9_dict = {'9a': ['20220906', '133000', '143000'], '9b': ['20220906', '153000', '161000']}
# case10_dict = {10: ['20220906', '161000', '180000']}
# case11_dict = {11: ['20220907', '130000', '134500']}
# case12_dict = {'12a': ['20220907', '134500', '151300'], '12b': ['20220907', '161800', '174500']}
# case13_dict = {13: ['20220907', '151300', '161800']}
# case14_dict = {'14a': ['20220914', '101000', '115500'], '14b': ['20220914', '134000', '142800']}
# case15_dict = {'15a': ['20220914','115500','131000'], '15b': ['20220914','142800','164500']}
# case16_dict = {16: ['20220916','143000','154000']}
# case17_dict = {17: ['20220916','164000','183500']}
# case18_dict = {'18a': ['20220922', '054000', '061500'], '18b': ['20220922', '063500', '073600'], '18c': ['20220922', '080000', '083000']}
# case20_dict = {20: ['20220926', '072000', '111500']}
# case21_dict = {21: ['20220929', '103500', '134500']}
# case22_dict = {22: ['20220930', '134800', '143000']}

eatl_isolated_case_dict = {1: ['20220909','161000','171000'], 2: ['20220909','173500','191000'], 3: ['20220910','190500','194200'],
                           4: ['20220910', '202500', '204500'], 5: ['20220916', '155500', '163500'], 6: ['20220920', '071000', '073500'],
                           7: ['20220920', '083000', '090000']}   #cases 1, 2, 3, 4, 5, 6, 7 above
eatl_organized_case_dict = {8: ['20220906', '110000', '120000'], '9a': ['20220906', '133000', '143000'], '9b': ['20220906', '153000', '161000'],
                            10: ['20220906', '161000', '180000'], 11: ['20220907', '130000', '134500'], '12a': ['20220907', '134500', '151300'],
                            '12b': ['20220907', '161800', '174500'], 13: ['20220907', '151300', '161800'], '14a': ['20220914', '101000', '115500'],
                            '14b': ['20220914', '134000', '142800'], '15a': ['20220914','115500','131000'], '15b': ['20220914','142800','164500'],
                            16: ['20220916','143000','154000'], 17: ['20220916','164000','183500'], '18a': ['20220922', '054000', '061500'],
                            '18b': ['20220922', '063500', '073600'], '18c': ['20220922', '080000', '083000'], 20: ['20220926', '072000', '111500'],
                            21: ['20220929', '103500', '134500'], 22: ['20220930', '134800', '143000']}   #cases 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22 above

#cases from eatl_xxx_case_dict above that were at least partly collected during growing or mature lifecycle stages
eatl_isolated_case_dict_growing_or_mature = {1: ['20220909','161000','171000'], 3: ['20220910','190500','194200'], 7: ['20220920', '083000', '090000']}   #cases 1, 3, 7 above
eatl_organized_case_dict_growing_or_mature = {8: ['20220906', '110000', '120000'], '9a': ['20220906', '133000', '143000'], '9b': ['20220906', '153000', '161000'],
                                              10: ['20220906', '161000', '180000'], '12a': ['20220907', '134500', '151300'], '12b': ['20220907', '161800', '174500'],
                                              13: ['20220907', '151300', '161800'], '15a': ['20220914','115500','131000'], '15b': ['20220914','142800','164500'],
                                              16: ['20220916','143000','154000'], 17: ['20220916','164000','183500'], '18a': ['20220922', '054000', '061500'],
                                              '18b': ['20220922', '063500', '073600'], '18c': ['20220922', '080000', '083000'], 20: ['20220926', '072000', '111500'],
                                              21: ['20220929', '103500', '134500'], 22: ['20220930', '134800', '143000']}   #cases 8, 9, 10, 12, 13, 15, 16, 17, 18, 20, 21, 22 above


height_edges = np.arange(1500, 8001, 500)
dbz_edges = np.arange(-20, 70.1, 5)
vel_edges = np.arange(-25, 25.1, 2)
# dbz_edges = np.arange(-20, 60.1, 5)
# vel_edges = np.arange(-13, 15.1, 2)


def CFAD(case_dict, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True):
    
    """"Calculate and plot Ku-band and Doppler Velocity CFAD 2-D arrays for 
        the given cases and their respective time ranges
    
    Parameters
    ----------
    case_dict:  dictionary of cases for which to calculate the total CFAD; 
                keys should be case numbers;
                values should be 3-element lists of case date and start/end times, 
                with date strings formatted as YYYYMMDD and time strings formatted as HHMMSS
    
    height_bin_edges:  a 1-D array of height [m] bin edges used to create the 2-D CFAD array/histogram    

    dbz_bin_edges:  a 1-D array of reflectivity [dBZ] bin edges used to create the 2-D CFAD array/histogram 

    vel_bin_edges:  a 1-D array of Doppler velocity [m/s] bin edges used to create the 2-D CFAD array/histogram       
                        
    normalized:  True/False; determines whether to normalize the CFAD array by maximum bin at any level 
                 (method from Zagrodnik et al., 2019) and create an additional, normalized CFAD plot
                        
    return:  2-D (normalized) CFAD arrays and plots """

    assert type(case_dict) == dict, "case_dict must be a dictionary"
    
    total_apr_profiles = 0 
    apr_profile_roll10 = 0
    new_ku_array = True
    new_vel_array = True
    
    for key in case_dict:
        print ('Processing Case {}...'.format(key))
        
        first_good_Ku_file_index = 0    #used to determine first usable Ku-band file for CAMP2Ex cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
        
        #grab the case's date and start/end times from the dictionary
        assert type(case_dict[key]) == list, "A key's values must be a list of date, start time, and end time" 
        desired_date = case_dict[key][0]
        if desired_date[:4] == '2019' and (key == 1 or key == 2 or key == 13 or key == '17b'):
            # #if a CAMP2Ex case spans across multiple days or the time range is fully the day after its science flight start date
            # case1_dict = {1: ['20190829',check code'234500','040000']}
            # case2_dict = {2: ['20190829',check code'041500','051000']}
            # case13_dict = {13: ['20191003', check code'231000', '004500']}
            # case17_dict = {'17a': ['20190915','223500','230500'], '17b': ['20190915',check code'031000','044500']}
            desired_date_next = datetime.strftime(datetime.strptime(desired_date, '%Y%m%d') + timedelta(days = 1), '%Y%m%d')
        else:
            desired_date_next = desired_date + ''
        start_time = case_dict[key][1]
        end_time = case_dict[key][2]
        #assert start_time < end_time, "Start time must precede end time in a key's list"
        
        #grab the ECCO-V convective-stratiform classification file for the given date
        use_file_list = []
        for file in os.listdir(os.path.join(os.getcwd(), 'ECCO-V_output', 'ECCO-V_classification_1D')):
            if desired_date in file:
                use_file_list.append(file)
        assert len(use_file_list) == 1, "Found either 0 or multiple ECCO-V files for the given date"
        use_file = use_file_list[0]
        
        ecco_filepath = os.path.join(os.getcwd(), 'ECCO-V_output', 'ECCO-V_classification_1D', use_file)
        ecco_df = pd.read_csv(ecco_filepath, sep = ',', dtype = str)
        ecco_df['Full_Datetime'] = pd.to_datetime(ecco_df['Year'].str.zfill(4) + ecco_df['Month'].str.zfill(2) + ecco_df['Day'].str.zfill(2) + ecco_df['Hour'].str.zfill(2) + ecco_df['Minute'].str.zfill(2) + ecco_df['Second'].astype(float).round().astype(int).astype(str), format = '%Y%m%d%H%M%S')
        
        #find the APR files of interest (for the desired date and time ranges)
        apr_folder = os.path.join(desired_date, 'APR_files')
        apr_file_list = sorted(os.listdir(apr_folder))
        
        #angles for each of the 24 rays in a given scan (used for ray adjustment; index order goes from left to right in the scan when looking ahead in the direction that the aircraft is headed)
        if desired_date[:4] == '2017':
            ray_angles = np.linspace(-25,25,24)[:-1]  #in degrees; omits 24th ray, which doesn't have data for Ku/Ka bands
        
            #find the list of APR files that have data for the desired time range
            apr_files_use = []
            first_file = 'blank'
            for file in apr_file_list:        #sorted() makes sure the code goes through the files in alphabetical order
                if file[0:3] == '.DS':         #delete possible .DS_Store files
                    os.remove(os.path.join(apr_folder, file))
                elif file[22:28] <= start_time:
                    first_file = file     #first_file will always be the file immediately before (or equal to) range_start
                elif file[22:28] >= end_time:
                    continue
                else:
                    if (first_file not in apr_files_use) and (first_file != 'blank'):
                        apr_files_use.append(first_file)
                    apr_files_use.append(file)
                    
            if apr_files_use == []:             #accounts for if start_time is greater than all of the file times, but still within the last file's time range; also accounts for start/end times equaling the times of adjacent files
                apr_files_use.append(first_file)
                
        elif desired_date[:4] == '2021':
            ray_angles = np.linspace(-25,25,25)
            
            #create a list of all the given day's desired range's APR files:    #for APR_plots.py
            start_time1 = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
            end_time1 = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
            
            for x in os.listdir(apr_folder):
                if x[0:3] == '.DS':         #delete hidden .DS_Store files if they come up (will show up if you delete a file)
                    os.remove(os.path.join(apr_folder, x))
            
            #find the starting APR file in apr_folder
            first_file_index = None       
            for i, x in enumerate(apr_file_list):
                file_start_time = datetime.strptime(x[13:21] + x[22:28], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
            
                if start_time1 <= file_start_time:  #if start_time1 is before the APR file start time and not within any previous APR file's time ranges
                    first_file_index = i
                    break
                elif (start_time1 >= file_start_time) and (start_time1 < file_end_time):
                    first_file_index = i
                    break
                else:
                    continue
            if first_file_index == None:
                sys.exit('Requested start_time is beyond all available APR files')
                
            #find the ending APR file in apr_folder
            last_file_index = None       
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[13:21] + x[22:28], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
            
                if end_time1 <= file_start_time:  #if end_time1 is before the APR file start time and not within any previous APR file's time ranges
                    last_file_index = i - 1
                    break
                elif (end_time1 > file_start_time) and (end_time1 <= file_end_time):
                    last_file_index = i
                    break
                else:
                    continue
            if last_file_index == None:  #the end_time is after all available APR files
                last_file_index = len(apr_file_list) - 1  #the last available APR file's index
            if last_file_index == -1:
                sys.exit('Requested end_time is before all available APR files')
            
            apr_files_use = apr_file_list[first_file_index:last_file_index + 1]
        
        elif desired_date[:4] == '2022':
            ray_angles = np.linspace(-25,25,25)
            
            #create a list of all the given day's desired range's APR files:    #for APR_plots.py
            start_time1 = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
            end_time1 = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
            
            #create a list of all the given day's desired range's APR files:
            for x in os.listdir(apr_folder):
                if x[0:3] == '.DS':         #delete hidden .DS_Store files if they come up (will show up if you delete a file)
                    os.remove(os.path.join(apr_folder, x))
            
            #find the starting APR file in apr_folder
            first_file_index = None 
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[46:54] + x[55:61], '%Y%m%d%H%M%S')
            
                if start_time1 <= file_start_time:  #if start_time is before the APR file start time and not within any previous APR file's time ranges
                    first_file_index = i
                    break
                elif (start_time1 >= file_start_time) and (start_time1 < file_end_time):
                    first_file_index = i
                    break
                else:
                    continue
            if first_file_index == None:
                sys.exit('Requested start_time is beyond all available APR files')
                
            #find the ending APR file in apr_folder
            last_file_index = None    
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[29:37] + x[38:44], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[46:54] + x[55:61], '%Y%m%d%H%M%S')
            
                if end_time1 <= file_start_time:  #if end_time is before the APR file start time and not within any previous APR file's time ranges
                    last_file_index = i - 1
                    break
                elif (end_time1 > file_start_time) and (end_time1 <= file_end_time):
                    last_file_index = i
                    break
                else:
                    continue
            if last_file_index == None:  #the end_time is after all available APR file ranges
                last_file_index = len(apr_file_list) - 1  #the last available APR file's index
            if last_file_index == -1:
                sys.exit('Requested end_time is before all available APR files')
            
            apr_files_use = apr_file_list[first_file_index:last_file_index + 1]
            
        elif desired_date[:4] == '2019':
            ray_angles = np.linspace(-25,25,25)
            
            #create a list of all the given day's desired range's APR files:    #for APR_plots.py
            if key == 2 or key == '17b':
                # #if a case spans across multiple days or the time range is fully the day after its science flight start date
                # case1_dict = {1: ['20190829',check code'234500','040000']}
                # case2_dict = {2: ['20190829',check code'041500','051000']}
                # case13_dict = {13: ['20191003', check code'231000', '004500']}
                # case17_dict = {'17a': ['20190915','223500','230500'], '17b': ['20190915',check code'031000','044500']}
                start_time1 = datetime.strptime(desired_date_next + start_time, '%Y%m%d%H%M%S')
            else:
                start_time1 = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
            
            end_time1 = datetime.strptime(desired_date_next + end_time, '%Y%m%d%H%M%S')
                
            #print (f'Case {key} start time: {start_time1}')  #sanity check to make sure the date ranges are correct
            #print (f'Case {key} end time: {end_time1}')      #sanity check to make sure the date ranges are correct
            
            #create a list of all the given day's desired range's APR files:
            for x in os.listdir(apr_folder):
                if x[0:3] == '.DS':         #delete hidden .DS_Store files if they come up (will show up if you delete a file)
                    os.remove(os.path.join(apr_folder, x))
            
            #find the starting APR file in apr_folder
            first_file_index = None
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):  
                file_start_time = datetime.strptime(x[35:43] + x[44:50], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[52:60] + x[61:67], '%Y%m%d%H%M%S')

                if start_time1 <= file_start_time:  #if start_time is before the APR file start time and not within any previous APR file's time ranges
                    first_file_index = i
                    break
                elif (start_time1 >= file_start_time) and (start_time1 < file_end_time):
                    first_file_index = i
                    break
                else:
                    continue
            if first_file_index == None:
                sys.exit('Requested start_time is after all available APR files')
                
            #find the ending APR file in apr_folder
            last_file_index = None 
            
            #sorted() makes sure the code goes through the files in alphabetical (chronological) order
            #GOING THROUGH THE FILES IN CHRONOLOGICAL ORDER IS ESSENTIAL FOR THIS CELL TO WORK PROPERLY!!
            for i, x in enumerate(apr_file_list):   
                file_start_time = datetime.strptime(x[35:43] + x[44:50], '%Y%m%d%H%M%S')
                file_end_time = datetime.strptime(x[52:60] + x[61:67], '%Y%m%d%H%M%S')
            
                if end_time1 <= file_start_time:  #if end_time is before the APR file start time and not within any previous APR file's time ranges
                    last_file_index = i - 1
                    break
                elif (end_time1 > file_start_time) and (end_time1 <= file_end_time):
                    last_file_index = i
                    break
                else:
                    continue
            if last_file_index == None:  #the end_time is after all available APR file ranges
                last_file_index = len(sorted(os.listdir(apr_folder))) - 1  #the last available APR file's index
            if last_file_index == -1:
                sys.exit('Requested end_time is before all available APR files')
                
            apr_files_use = apr_file_list[first_file_index:last_file_index + 1]
        
        else:
            sys.exit('Not a CPEX or CPEX-AW or CPEX-CV or CAMP2Ex case')
    
        #print (apr_files_use)  #sanity check to make sure you grabbed the right files


        for apr_filepath in apr_files_use:
            
            if desired_date[:4] == '2017' or desired_date[:4] == '2021':    #CPEX(-AW)
            
                #Low resolution ('lores') radar variables in APR hdf files
                ku_band = 'zhh14' #Ku-band reflectivity
                vel = 'vel14c' #mean Doppler Velocity dealiased and from Ku&Ka band
            
                apr_file = h5py.File(os.path.join(apr_folder, apr_filepath), 'r')
                
                if ('lores' in apr_file.keys()) and (ku_band in apr_file['lores'].keys()):
                    
                    try:   #some CPEX-AW APR files have corrupted Ku-band data; if so, skip the Ku-band for 
                           #that file (corrupted: "OSError: Can't read data (inflate() failed)")
                        ku_data = apr_file['lores'][ku_band][:]
                    except:
                        apr_file.close()
                        continue  #both Ku-band and velocity CFADs rely on Ku-band data availability
    
                    try:   #some CPEX-AW APR files have corrupted velocity data; if so, skip the Ku-band for 
                           #that file (corrupted: "OSError: Can't read data (inflate() failed)")
                        vel_data = apr_file['lores'][vel][:]
                        vel_good = True
                    except:
                        vel_good = False                    
                
                    #grab the radar variables of interest
                    time = apr_file['lores']['scantime'][:]
                    alt3d = apr_file['lores']['alt3D'][:]
                    roll = apr_file['lores']['roll'][:]
                    
                    if apr_filepath == apr_files_use[0] or apr_filepath == apr_files_use[-1]:
                    #the complete APR file time range may not need to be used, so need to locate the closest time (and corresponding index) to the desired start/end time
                        
                        #Convert APR times to datetimes
                        time_dates = np.empty(time.shape[1], dtype=object)
                        for j in np.arange(0, time.shape[1]):
                            #tmp = datetime(time[12,j])
                            tmp = datetime.utcfromtimestamp(time[12,j])  #12 could be any ray, as it is the ray number and each ray of a given scan has the same time
                            time_dates[j] = tmp
                        
                        unique_apr_times = time_dates  #all the times in the given APR file
                        
                        if len(apr_files_use) == 1:  #i.e. apr_filepath == apr_files_use[0] and apr_filepath == apr_files_use[-1]
                        
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                        
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                                    
                        elif apr_filepath == apr_files_use[0]:  #i.e. the first APR file in the apr_files_use list
                            end_time_idx = time.shape[1] - 1
                            
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                                    
                        else:  #apr_filepath == apr_files_use[-1]  #i.e. the last APR file in the apr_files_use list
                            start_time_idx = 0
                            
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))                
                            
                    else:  #the entire APR file is within the desired time range, so set the start/end indices to the first/last indices of the APR file
                        start_time_idx = 0
                        end_time_idx = time.shape[1] - 1
                            
                    #numpy.histogram2d(dBZ, z) (Zagrodnik et al., 2019) CFAD method:

                    #loop through the given APR file’s valid times/scans/profiles
                    for time_idx in range(start_time_idx, end_time_idx + 1):          #time_idx represents the scan number
                    
                        profile_time = datetime.utcfromtimestamp(time[12, time_idx])  #12 could be any ray, as it is the ray number and each ray of a given scan has the same time
                        ecco_ip = np.argmin(abs(ecco_df['Full_Datetime'] - profile_time))  #abs() necessary to properly subtract datetime objects
                                                                                            #returns the integer position of the minimum value
                        #print (ecco_df['Full_Datetime'].iloc[ecco_index])           #to check that the correct time was chosen
                        classification = ecco_df['Classification'].iloc[ecco_ip]     #grab the classification of the ECCO-V time closest to the given profile time (may/should be the exact time)
                        
                        if classification not in ['0', '1', '2']:
                            raise ValueError(f'{classification} is not a valid ECCO-V classification.')
                    
                        # #skip profiles with an ECCO-V classification of "no classification" (0) or "stratiform" (1)
                        # #i.e., only use "convective" (2) profiles
                        # if classification == '0' or classification == '1':
                        #     continue
                        
                        #skip profiles with an ECCO-V classification of "no classification" (0) or "convective" (2)
                        #i.e., only use "stratiform" (1) profiles
                        if classification == '0' or classification == '2':
                            continue
                        
                        total_apr_profiles += 1
                        
                        #choose the "nadir" ray factoring in aircraft roll
                        ac_roll = np.nanmean(roll[:,time_idx])  #roll varies slightly w/ray, so take the average roll value for a given scan and use that for ray adjustment
                        ray_use = np.argmin(np.abs(ray_angles - ac_roll))  #the index of the ray whose angle is closest to that of ac_roll
                        
                        if abs(ac_roll) >= 10:
                            apr_profile_roll10 += 1
                        
                        prof_height = alt3d[:,ray_use,time_idx]
                        prof_dbz = ku_data[:,ray_use,time_idx]
                        
                        #grab the height/dbz data above 1.5km
                        idx_1500m = np.argmin(np.abs(prof_height - 1500))
                        prof_height_above1500m = np.flip(prof_height[:idx_1500m + 1])  #height profile above 1.5km, in ascending order
                        prof_dbz_above1500m = np.flip(prof_dbz[:idx_1500m + 1])  #Ku-band profile above 1.5km, in ascending order
                        
                        #store all the profile's height/dbz data in long 1-D concatenated arrays
                        if new_ku_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                            height_concat = prof_height.copy()
                            dbz_concat = prof_dbz.copy()
                            
                            #create a DataFrame of values above 1.5km, in order to calculate meadian/quartile reflectivity profile for the given case
                                #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                    #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile     
                            height_profiles_above1500m_df = pd.DataFrame(prof_height_above1500m)  #height profile from 1500m to top of profile
                            dbz_profiles_above1500m_df = pd.DataFrame(prof_dbz_above1500m)        #dbz profile from 1500m to top of profile
                            new_ku_array = False
                        else:
                            height_concat = np.concatenate((height_concat, prof_height))
                            dbz_concat = np.concatenate((dbz_concat, prof_dbz))
                            
                            #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
                            height_profiles_above1500m_df = pd.concat((height_profiles_above1500m_df, pd.Series(prof_height_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                            dbz_profiles_above1500m_df = pd.concat((dbz_profiles_above1500m_df, pd.Series(prof_dbz_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                        
                        # if vel_good:
                        #     prof_vel = vel_data[:,ray_use,time_idx]
                            
                        #     #store all the profile's velocity data in long 1-D concatenated arrays if 
                        #     #the profile has Ku data > 0 dBZ above 1.5km (i.e. omits clear profiles)
                            
                        #     if np.nanmax(prof_dbz_above1500m) > 0:
                        #         if new_vel_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                        #             vel_concat = prof_vel.copy()
                        #             height_vel_concat = prof_height.copy()
                        #             new_vel_array = False
                        #         else:
                        #             vel_concat = np.concatenate((vel_concat, prof_vel))
                        #             height_vel_concat = np.concatenate((height_vel_concat, prof_height))
                        #     else:  #clear profiles (which may have noisy velocity data) are omitted from the velocity CFAD
                        #         pass
                        # else:
                        #     pass
                else:
                    pass
                    
                apr_file.close()
                
            elif desired_date[:4] == '2022':    #CPEX-CV
            
                ku_band = 'lores_zhh14' #Ku-band reflectivity
                vel = 'lores_vel14c' #Mean Doppler Velocity from Ku-band (surface Doppler velocity is subtracted and free of aliasing)
            
                apr_file = xr.open_dataset(os.path.join(apr_folder, apr_filepath))
                
                if ku_band in apr_file.keys():
                    
                    try:   #some APR files, at least in the preliminary data, have corrupted Ku-band data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        ku_data = apr_file[ku_band][:]
                    except:
                        first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                        apr_file.close()
                        continue  #both Ku-band and velocity CFADs rely on Ku-band data availability
    
                    try:   #some APR files, at least in the preliminary data, have corrupted velocity data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        vel_data = apr_file[vel][:]
                        vel_good = True
                    except:
                        vel_good = False                    
                
                    #grab the radar variables of interest
                    time = apr_file['time'][:]          #For 'lores': Time of scan, in seconds since midnight UTC of [YYYY-mm-DD]
                    alt3d = apr_file['lores_alt3D'][:]
                    roll = apr_file['lores_roll'][:]
                    
                    if apr_filepath == apr_files_use[first_good_Ku_file_index] or apr_filepath == apr_files_use[-1]:
                    #the complete APR file time range may not need to be used, so need to locate the closest time (and corresponding index) to the desired start/end time
                        
                        #Convert APR times to datetimes
                        time_dates = np.empty(time.shape, dtype=object)
                        for i in np.arange(0, time.shape[0]):
                            #hour, second automatically revert to midnight (hour = 0, seconds = 0) for '%Y%m%d'
                            time_dates[i] = datetime.strptime(desired_date, '%Y%m%d') + timedelta(seconds = float(time[i].values))
                        
                        unique_apr_times = time_dates[:]  #all the times in the given APR file
                        
                        if len(apr_files_use) == 1:  #i.e. apr_filepath == apr_files_use[first_good_Ku_file_index] and apr_filepath == apr_files_use[-1]
                        
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                        
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                                    
                        elif apr_filepath == apr_files_use[first_good_Ku_file_index]:  #i.e. the first APR file in the apr_files_use list
                            end_time_idx = time.shape[0] - 1
                            
                            #find the closest time (and its corresponding index) to the desired start_time
                            desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                                    
                        else:  #apr_filepath == apr_files_use[-1]  #i.e. the last APR file in the apr_files_use list
                            start_time_idx = 0
                            
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))                
                            
                    else:  #the entire APR file is within the desired time range, so set the start/end indices to the first/last indices of the APR file
                        start_time_idx = 0
                        end_time_idx = time.shape[0] - 1
                            
                    #numpy.histogram2d(dBZ, z) (Zagrodnik et al., 2019) CFAD method:
                    
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                       
                    
                    #loop through the given APR file’s valid times/scans/profiles
                    for time_idx in range(start_time_idx, end_time_idx + 1):          #time_idx represents the scan number
                        
                        profile_time = datetime.strptime(desired_date, '%Y%m%d') + timedelta(seconds = float(time[time_idx].values))
                        ecco_ip = np.argmin(abs(ecco_df['Full_Datetime'] - profile_time))  #abs() necessary to properly subtract datetime objects
                                                                                            #returns the integer position of the minimum value
                        #print (ecco_df['Full_Datetime'].iloc[ecco_index])           #to check that the correct time was chosen
                        classification = ecco_df['Classification'].iloc[ecco_ip]     #grab the classification of the ECCO-V time closest to the given profile time (may/should be the exact time)
                        
                        if classification not in ['0', '1', '2']:
                            raise ValueError(f'{classification} is not a valid ECCO-V classification.')
                    
                        # #skip profiles with an ECCO-V classification of "no classification" (0) or "stratiform" (1)
                        # #(i.e., only use "convective" (2) profiles)
                        # if classification == '0' or classification == '1':
                        #     continue
                        
                        #skip profiles with an ECCO-V classification of "no classification" (0) or "convective" (2)
                        #(i.e., only use "stratiform" (1) profiles)
                        if classification == '0' or classification == '2':
                            continue
                        
                        total_apr_profiles += 1
                        
                        #choose the "nadir" ray factoring in aircraft roll
                        ac_roll = np.nanmean(roll[time_idx,:])  #roll varies slightly w/ray, so take the average roll value for a given scan and use that for ray adjustment
                        ray_use = np.argmin(np.abs(ray_angles - ac_roll))  #the index of the ray whose angle is closest to that of ac_roll; i.e., the "pseudo-nadir" ray
                        
                        if abs(ac_roll) >= 10:
                            apr_profile_roll10 += 1
                        
                        prof_height = alt3d[time_idx,ray_use,:].values
                        prof_dbz = ku_data[time_idx,ray_use,:].values
                        
                        #grab the height/dbz data above 1.5km
                        idx_1500m = np.argmin(np.abs(prof_height - 1500))
                        prof_height_above1500m = np.flip(prof_height[:idx_1500m + 1])  #height profile above 1.5km, in ascending order
                        prof_dbz_above1500m = np.flip(prof_dbz[:idx_1500m + 1])  #Ku-band profile above 1.5km, in ascending order
                        
                        #store all the profile's height/dbz data in long 1-D concatenated arrays
                        if new_ku_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                            height_concat = prof_height.copy()
                            dbz_concat = prof_dbz.copy()
                            
                            #create a DataFrame of values above 1.5km, in order to calculate median/quartile reflectivity profile for the given case
                                #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                    #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile     
                            height_profiles_above1500m_df = pd.DataFrame(prof_height_above1500m)  #height profile from 1500m to top of profile
                            dbz_profiles_above1500m_df = pd.DataFrame(prof_dbz_above1500m)        #dbz profile from 1500m to top of profile
                            new_ku_array = False
                        else:
                            height_concat = np.concatenate((height_concat, prof_height))
                            dbz_concat = np.concatenate((dbz_concat, prof_dbz))
                            
                            #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
                            height_profiles_above1500m_df = pd.concat((height_profiles_above1500m_df, pd.Series(prof_height_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                            dbz_profiles_above1500m_df = pd.concat((dbz_profiles_above1500m_df, pd.Series(prof_dbz_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                        
                        # if vel_good:
                        #     prof_vel = vel_data[time_idx,ray_use,:].values
                            
                        #     #store all the profile's velocity data in long 1-D concatenated arrays if 
                        #     #the profile has Ku data > 0 dBZ above 1.5km (i.e. omits clear profiles)
                            
                        #     if np.nanmax(prof_dbz_above1500m) > 0:
                        #         if new_vel_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                        #             vel_concat = prof_vel.copy()
                        #             height_vel_concat = prof_height.copy()
                        #             new_vel_array = False
                        #         else:
                        #             vel_concat = np.concatenate((vel_concat, prof_vel))
                        #             height_vel_concat = np.concatenate((height_vel_concat, prof_height))
                        #     else:  #clear profiles (which may have noisy velocity data) are omitted from the velocity CFAD
                        #         pass
                        # else:
                        #     pass
                            
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                            
                            
                else:
                    first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                    
                apr_file.close()
                               
            elif desired_date[:4] == '2019':    #CAMP2Ex
            
                ku_band = 'lores_zhh14' #Ku-band reflectivity, (Scan, Ray, Range Bin)
                vel = 'lores_vel14c'    #Mean Doppler Velocity at Ku-band CORRECTED by surface-reference technique (surface Doppler velocity is subtracted and free of aliasing), (Scan, Ray, Range Bin)
            
                apr_file = xr.open_dataset(os.path.join(apr_folder, apr_filepath))
                file_start_date = apr_filepath[35:43]  #this date determines how the 'Time' variable is calculated (will be seconds since midnight UTC of this date (YYYMMMDD))
                
                if ku_band in apr_file.keys():
                    
                    try:   #some APR files, at least in the preliminary data, have corrupted Ku-band data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        ku_data = apr_file[ku_band][:]
                    except:
                        first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                        apr_file.close()
                        continue  #both Ku-band and velocity CFADs rely on Ku-band data availability
    
                    try:   #some APR files, at least in the preliminary data, have corrupted velocity data; 
                           #if so, skip plotting the Ku-band for that file 
                           #corrupted: "OSError: Can't read data (inflate() failed)"
                        vel_data = apr_file[vel][:]
                        vel_good = True
                    except:
                        vel_good = False                    
                
                    #grab the radar variables of interest
                    time = apr_file['time'][:]          #For 'lores': Time of scan, in seconds since midnight UTC of file_start_date (YYYMMMDD)
                    alt3d = apr_file['lores_alt3D'][:]
                    roll = apr_file['lores_roll'][:]
                   
                    if apr_filepath == apr_files_use[first_good_Ku_file_index] or apr_filepath == apr_files_use[-1]:
                    #the complete APR file time range may not need to be used, so need to locate the closest time (and corresponding index) to the desired start/end time
                        
                        #Convert APR times to datetimes
                        time_dates = np.empty(time.shape, dtype=object)
                        for i in np.arange(0, time.shape[0]):
                            #hour, second automatically revert to midnight (hour = 0, seconds = 0) for '%Y%m%d'
                            time_dates[i] = datetime.strptime(file_start_date, '%Y%m%d') + timedelta(seconds = float(time[i].values))
                        
                        unique_apr_times = time_dates[:]  #all the times in the given APR file
                     
                        if len(apr_files_use) == 1:  #i.e. apr_filepath == apr_files_use[first_good_Ku_file_index] and apr_filepath == apr_files_use[-1]
                        
                            #find the closest time (and its corresponding index) to the desired start_time
                            if key == 2 or key == '17b':
                                desired_start_time = datetime.strptime(desired_date_next + start_time, '%Y%m%d%H%M%S')
                            else:
                                desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                        
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date_next + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                                    
                        elif apr_filepath == apr_files_use[first_good_Ku_file_index]:  #i.e. the first APR file in the apr_files_use list
                            end_time_idx = time.shape[0] - 1
                            
                            #find the closest time (and its corresponding index) to the desired start_time
                            if key == 2 or key == '17b':
                                desired_start_time = datetime.strptime(desired_date_next + start_time, '%Y%m%d%H%M%S')
                            else:
                                desired_start_time = datetime.strptime(desired_date + start_time, '%Y%m%d%H%M%S')
                            start_time_idx = np.argmin(abs(unique_apr_times - desired_start_time))
                                    
                        else:  #apr_filepath == apr_files_use[-1]  #i.e. the last APR file in the apr_files_use list
                            start_time_idx = 0
                            
                            #find the closest time (and its corresponding index) to the desired end_time
                            desired_end_time = datetime.strptime(desired_date_next + end_time, '%Y%m%d%H%M%S')
                            end_time_idx = np.argmin(abs(unique_apr_times - desired_end_time))
                            
                    else:  #the entire APR file is within the desired time range, so set the start/end indices to the first/last indices of the APR file
                        start_time_idx = 0
                        end_time_idx = time.shape[0] - 1
                            
                    #numpy.histogram2d(dBZ, z) (Zagrodnik et al., 2019) CFAD method:
                    
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                       
                    
                    #loop through the given APR file’s valid times/scans/profiles
                    for time_idx in range(start_time_idx, end_time_idx + 1):          #time_idx represents the scan number
                        
                        profile_time = datetime.strptime(file_start_date, '%Y%m%d') + timedelta(seconds = float(time[time_idx].values))
                        ecco_ip = np.argmin(abs(ecco_df['Full_Datetime'] - profile_time))  #abs() necessary to properly subtract datetime objects
                                                                                            #returns the integer position of the minimum value
                        #print (ecco_df['Full_Datetime'].iloc[ecco_index])           #to check that the correct time was chosen
                        classification = ecco_df['Classification'].iloc[ecco_ip]     #grab the classification of the ECCO-V time closest to the given profile time (may/should be the exact time)
                        
                        if classification not in ['0', '1', '2']:
                            raise ValueError(f'{classification} is not a valid ECCO-V classification.')
                    
                        # #skip profiles with an ECCO-V classification of "no classification" (0) or "stratiform" (1)
                        # #(i.e., only use "convective" (2) profiles)
                        # if classification == '0' or classification == '1':
                        #     continue
                        
                        #skip profiles with an ECCO-V classification of "no classification" (0) or "convective" (2)
                        #(i.e., only use "stratiform" (1) profiles)
                        if classification == '0' or classification == '2':
                            continue
                        
                        total_apr_profiles += 1
                        
                        #choose the "nadir" ray factoring in aircraft roll
                        ac_roll = np.nanmean(roll[time_idx,:])  #roll varies slightly w/ray, so take the average roll value for a given scan and use that for ray adjustment
                        ray_use = np.argmin(np.abs(ray_angles - ac_roll))  #the index of the ray whose angle is closest to that of ac_roll; i.e., the "pseudo-nadir" ray
                        
                        if abs(ac_roll) >= 10:
                            apr_profile_roll10 += 1
                        
                        prof_height = alt3d[time_idx,ray_use,:].values
                        prof_dbz = ku_data[time_idx,ray_use,:].values
                        
                        #grab the height/dbz data above 1.5km
                        idx_1500m = np.argmin(np.abs(prof_height - 1500))
                        prof_height_above1500m = np.flip(prof_height[:idx_1500m + 1])  #height profile above 1.5km, in ascending order
                        prof_dbz_above1500m = np.flip(prof_dbz[:idx_1500m + 1])  #Ku-band profile above 1.5km, in ascending order
                        
                        #store all the profile's height/dbz data in long 1-D concatenated arrays
                        if new_ku_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                            height_concat = prof_height.copy()
                            dbz_concat = prof_dbz.copy()
                            
                            #create a DataFrame of values above 1.5km, in order to calculate median/quartile reflectivity profile for the given case
                                #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                    #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile     
                            height_profiles_above1500m_df = pd.DataFrame(prof_height_above1500m)  #height profile from 1500m to top of profile
                            dbz_profiles_above1500m_df = pd.DataFrame(prof_dbz_above1500m)        #dbz profile from 1500m to top of profile
                            new_ku_array = False
                        else:
                            height_concat = np.concatenate((height_concat, prof_height))
                            dbz_concat = np.concatenate((dbz_concat, prof_dbz))
                            
                            #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
                                #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
                            height_profiles_above1500m_df = pd.concat((height_profiles_above1500m_df, pd.Series(prof_height_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                            dbz_profiles_above1500m_df = pd.concat((dbz_profiles_above1500m_df, pd.Series(prof_dbz_above1500m)), axis = 1, ignore_index = True)  #add profile as a new column to the df; differences in profile lengths are filled in with NaNs
                        
                        # if vel_good:
                        #     prof_vel = vel_data[time_idx,ray_use,:].values
                            
                        #     #store all the profile's velocity data in long 1-D concatenated arrays if 
                        #     #the profile has Ku data > 0 dBZ above 1.5km (i.e. omits clear profiles)
                            
                        #     if np.nanmax(prof_dbz_above1500m) > 0:
                        #         if new_vel_array:  #if this is the first qualifying time and ray of the first APR file of the first case in case_dict
                        #             vel_concat = prof_vel.copy()
                        #             height_vel_concat = prof_height.copy()
                        #             new_vel_array = False
                        #         else:
                        #             vel_concat = np.concatenate((vel_concat, prof_vel))
                        #             height_vel_concat = np.concatenate((height_vel_concat, prof_height))
                        #     else:  #clear profiles (which may have noisy velocity data) are omitted from the velocity CFAD
                        #         pass
                        # else:
                        #     pass
                            
########################################################################################################################### 
###########################################################################################################################
###########################################################################################################################
###########################################################################################################################                            
                            
                else:
                    first_good_Ku_file_index += 1    #used to determine first usable Ku-band file for CPEX-CV cases below; without this, an error is brought up due to the first APR file in apr_files_use only being W-band data (i.e., _Wn.nc)
                    
                apr_file.close()
                
            else:
                sys.exit('Not a CPEX or CPEX-AW or CPEX-CV or CAMP2Ex case')
                
        print ('Case {} complete'.format(key))
       
    #calculate median/quantile dBZ profiles for the given case
    
    #replace blank data (-99.99) with NaNs
    dbz_profiles_above1500m_df[dbz_profiles_above1500m_df <= -99] = np.nan
    height_profiles_above1500m_df[height_profiles_above1500m_df < 0] = np.nan
    
    #convert all the dropsonde's Ku-band data from dBZ to mm^6 m^-3
    Z_df = 10**(dbz_profiles_above1500m_df / 10)
    
    #calculate the dropsonde's median/quantile heights and reflectivity (in mm^6 m^-3) and standard deviation at each height level
    #NOTE: WE ARE ASSUMING THAT EACH ROW CORRESPONDS TO A NEAR IDENTICAL HEIGHT LEVEL
        #this is the best we can do, and should be fine given that each profile is starting from ~1500m and the height resolution is the same for each profile
        
    median_height_profile_above1500m = height_profiles_above1500m_df.median(axis = 1)  #axis of 1 = across all columns (so for each row); NaNs are ignored by default in Pandas
    Z_median = Z_df.median(axis = 1)  #axis of 1 = across all columns (so for each row); NaNs are ignored by default in Pandas
    Z_Q1 = Z_df.quantile(q = 0.25, axis = 1)
    Z_Q3 = Z_df.quantile(q = 0.75, axis = 1)
    #Z_std = Z_df.std(axis = 1)
    
    #convert the dropsonde's median/quartile reflectivity and standard deviation profile back to dBZ
    medianDBZ_profile_above1500m = 10 * np.log10(Z_median)
    Q1DBZ_profile_above1500m = 10 * np.log10(Z_Q1)
    Q3DBZ_profile_above1500m = 10 * np.log10(Z_Q3)
    #stdDBZ_profile_above1500m = 10 * np.log10(Z_std)      
        
    
    #setting the height, reflectivity, and velocity bin edges, along with their associated meshgrids
    height_edges = height_bin_edges
    dbz_edges = dbz_bin_edges
    #dbz_meshgrid, height_meshgrid = np.meshgrid(dbz_edges, height_edges)
    
    vel_edges = vel_bin_edges
    #vel_meshgrid, height_vel_meshgrid = np.meshgrid(vel_edges, height_edges)
    
    #setting the height, reflectivity, and velocity bin centers and their associated meshgrids for contourf plotting
    height_centers = (height_edges[:-1] + height_edges[1:]) / 2    
    dbz_centers = (dbz_edges[:-1] + dbz_edges[1:]) / 2
    dbz_centers_meshgrid, height_centers_meshgrid = np.meshgrid(dbz_centers, height_centers)
    
    vel_centers = (vel_edges[:-1] + vel_edges[1:]) / 2
    vel_centers_meshgrid, height_vel_centers_meshgrid = np.meshgrid(vel_centers, height_centers)
  
    #create the 2D histogram of values/frequencies for Ku-band data
    cfad_array, xedges, yedges = np.histogram2d(dbz_concat, height_concat, bins = (dbz_edges,height_edges))
    
    #transpose cfad_array shape (rows, columns) to be (height, dbz) instead of (dbz, height)
    cfad_array = cfad_array.T
    #cfad_array = np.log10(cfad_array)  #creates log-weighted CFADs
    contours = np.linspace(0, np.nanmax(cfad_array), 21)
    colorbar_label = 'Total Frequency [#]'
    var_label = 'Ku-band Reflectivity [dBZ]'
    save_label = 'CFAD_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Ku.png'
    plot_CFAD(cfad_array, contours, colorbar_label, save_label, var_label, dbz_centers_meshgrid, height_centers_meshgrid, dbz_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = True)
    
    #normalize the CFAD and make a normalized CFAD plot if that is also desired
    if normalize:
        #cfad_array = cfad_array / np.nanmax(cfad_array) * 100
        height_bin_maxs = np.nanmax(cfad_array, axis = 1)   #normalize the CFAD by max count in each height bin
        cfad_array = cfad_array / height_bin_maxs[:, np.newaxis] * 100
        contours = np.arange(0,101,5)
        colorbar_label = 'Normalized Frequency [%]'
        save_label = 'CFADnorm_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Ku.png'
        plot_CFAD(cfad_array, contours, colorbar_label, save_label, var_label, dbz_centers_meshgrid, height_centers_meshgrid, dbz_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = True)
        
        
    # #create the 2D histogram of values/frequencies for Doppler Velocity data
    # cfad_array_vel, xedges, yedges = np.histogram2d(vel_concat, height_vel_concat, bins = (vel_edges,height_edges))
    
    # #transpose cfad_array shape (rows, columns) to be (height, vel) instead of (vel, height)
    # cfad_array_vel = cfad_array_vel.T
    # #cfad_array_vel = np.log10(cfad_array_vel)  #creates log-weighted CFADs
    # contours = np.linspace(0, np.nanmax(cfad_array_vel), 21)
    # colorbar_label = 'Total Frequency [#]'
    # var_label = 'Mean Doppler Velocity [m/s]'
    # save_label = 'CFAD_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Vel.png'
    # plot_CFAD(cfad_array_vel, contours, colorbar_label, save_label, var_label, vel_centers_meshgrid, height_vel_centers_meshgrid, vel_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = False)
    
    # #normalize the CFAD and make a normalized CFAD plot if that is also desired
    # if normalize:
    #     #cfad_array_vel = cfad_array_vel / np.nanmax(cfad_array_vel) * 100
    #     height_bin_maxs = np.nanmax(cfad_array_vel, axis = 1)   #normalize the CFAD by max count in each height bin
    #     cfad_array_vel = cfad_array_vel / height_bin_maxs[:, np.newaxis] * 100
    #     contours = np.arange(0,101,5)
    #     colorbar_label = 'Normalized Frequency [%]'
    #     save_label = 'CFADnorm_Cases' + '-'.join(map(str, list(case_dict.keys()))) + '_Vel.png'
    #     plot_CFAD(cfad_array_vel, contours, colorbar_label, save_label, var_label, vel_centers_meshgrid, height_vel_centers_meshgrid, vel_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = False)
        
    # #print ('Percent of profiles with A/C roll >= 10 degrees:', apr_profile_roll10 / total_apr_profiles * 100)
    
    # return cfad_array, cfad_array_vel, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m
    return cfad_array, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m

def plot_CFAD(cfad_array, contours, colorbar_label, save_label, var_label, var_centers_meshgrid, height_centers_meshgrid, var_edges, case_dict, median_height_profile_above1500m, medianDBZ_profile_above1500m, Q1DBZ_profile_above1500m, Q3DBZ_profile_above1500m, dbz_plot = True):
    
    """Plot a contourf CFAD given a CFAD 2-D array, contour levels, plot/image labels, and variable/height meshgrids"""
    
    #plot the CFAD
    fig, ax = plt.subplots(1,1, figsize=(21,21))
    cmap = mplc.ListedColormap(['#ffffff', '#d8fcfa', '#bef5f6', '#aaedf1', '#98e4ec', '#89dae7', '#7cd0e2', 
                                '#70c6dd', '#65bcd9', '#5cb2d4', '#53a8cf', '#4b9dca', '#4393c5', '#3c89c0', 
                                '#357ebb', '#2f74b6', '#286ab1', '#2160ac', '#1956a7', '#0f4ca2', '#00429d'])
    
    #the cmap below omits the white fill at the beginnning
    # cmap = mplc.ListedColormap(['#d8fcfa', '#bef5f6', '#aaedf1', '#98e4ec', '#89dae7', '#7cd0e2', 
    #                             '#70c6dd', '#65bcd9', '#5cb2d4', '#53a8cf', '#4b9dca', '#4393c5', '#3c89c0', 
    #                             '#357ebb', '#2f74b6', '#286ab1', '#2160ac', '#1956a7', '#0f4ca2', '#00429d'])
    
    #cs = ax.pcolormesh(dbz_meshgrid, height_meshgrid, cfad_array, cmap = cmap)
    #cs = ax.contour(dbz_centers_meshgrid, height_centers_meshgrid, cfad_array, levels = contours, cmap = cmap, linewidths = 1)
    cs = ax.contourf(var_centers_meshgrid, height_centers_meshgrid / 1000, cfad_array, levels = contours, cmap = cmap)
    if dbz_plot:
        ax.plot(medianDBZ_profile_above1500m, median_height_profile_above1500m / 1000, color = 'k', linestyle = '-', linewidth = 5, label = 'Case ' + ','.join(map(str, list(case_dict.keys()))) + ' Median Profile') 
        ax.plot(Q1DBZ_profile_above1500m, median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5, label = 'Case ' + ','.join(map(str, list(case_dict.keys()))) + ' Q1 and Q3 Profiles')
        ax.plot(Q3DBZ_profile_above1500m, median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5)
        ax.legend(loc = 'upper right')
    ax.set_ylabel('Altitude [m]', fontsize=30, fontweight = 'bold')
    ax.set_xlabel(var_label, fontsize=30, fontweight = 'bold')
    ax.tick_params(length = 15, width = 5, labelsize = 25)
    #ax.set_title('Total CFAD for Case ' + ','.join(map(str, list(case_dict.keys()))), fontsize=35, fontweight = 'bold')
    ax.set_title('Case ' + ','.join(map(str, list(case_dict.keys()))) + ' Normalized CFAD (Only Convective Regions)', fontsize=35, fontweight = 'bold')
    ax.set_ylim([(height_edges[0] + 250) / 1000, (height_edges[-1] - 250) / 1000])
    #ax.set_xlim([var_edges[0],var_edges[-1]])
    ax.set_xlim([5,60])  #for Ku-band
    
    #set the colorbar axis
    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.08,
                       ax.get_position().x1-ax.get_position().x0, 0.02])    #Left, bottom, width, height (all [0,1])

    #create the colorbar
    cbar = plt.colorbar(cs, cax = cax, orientation = 'horizontal')
    cbar.ax.tick_params(length = 10, width = 3, labelsize = 23)
    cbar.set_label(label = colorbar_label, fontsize = 30, fontweight = 'bold')
    
    #save the figure
    plt.savefig(''.join(['/Users/ben/Desktop/', save_label]), bbox_inches = 'tight')
    plt.close()


def difference_CFAD(case_dict_1, case_dict_2, height_bin_edges, dbz_bin_edges, vel_bin_edges, diff_plot_title, save_label_manual):

    """"Calculate and plot Ku-band and Doppler Velocity difference (normalized) CFAD 2-D arrays for 
        the given pair of dictionaries containing given cases and their respective time ranges
    
    Parameters
    ----------
    case_dict_1:  dictionary of cases for which to calculate the first normalized CFAD; 
                  keys should be case numbers;
                  values should be 3-element lists of case date and start/end times, 
                  with date strings formatted as YYYYMMDD and time strings formatted as HHMMSS
                  
    case_dict_2:  dictionary of cases for which to calculate the second normalized CFAD; 
                  keys should be case numbers;
                  values should be 3-element lists of case date and start/end times, 
                  with date strings formatted as YYYYMMDD and time strings formatted as HHMMSS
    
    height_bin_edges:  a 1-D array of height [m] bin edges used to create the 2-D difference CFAD array/histogram    

    dbz_bin_edges:  a 1-D array of reflectivity [dBZ] bin edges used to create the 2-D difference CFAD array/histogram 

    vel_bin_edges:  a 1-D array of Doppler velocity [m/s] bin edges used to create the 2-D difference CFAD array/histogram
                        
    return:  2-D (normalized) difference CFAD plots (case_dict_2 - case_dict_1)"""

    #calculate the Ku-band and Doppler Velocity 2-D normalized CFADs for each case
    # first_dbz_CFAD, first_vel_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m = CFAD(case_dict_1, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    # second_dbz_CFAD, second_vel_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m = CFAD(case_dict_2, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    first_dbz_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m = CFAD(case_dict_1, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    second_dbz_CFAD, dbz_centers_meshgrid, height_centers_meshgrid, vel_centers_meshgrid, height_vel_centers_meshgrid, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m = CFAD(case_dict_2, height_bin_edges, dbz_bin_edges, vel_bin_edges, normalize = True)
    
    #calculate the Ku-band and Doppler Velocity 2-D difference CFADs
    dbz_diff_CFAD = second_dbz_CFAD - first_dbz_CFAD
    # vel_diff_CFAD = second_vel_CFAD - first_vel_CFAD
    
    #calculate the maximum magnitude for each CFAD and create contours for diverging colormaps accordingly
    
    #dbz_highest_mag = np.nanmax(np.abs(dbz_diff_CFAD))  #creates unique colorbar range for each difference CFAD plot
    dbz_highest_mag = 100  #creates uniform colorbar range across all difference CFAD plots
    dbz_contours = np.linspace(-dbz_highest_mag, dbz_highest_mag, 41)  #an odd number of intervals (21) guarantees a contour at 0 for a range from -x to x
    
    # #vel_highest_mag = np.nanmax(np.abs(vel_diff_CFAD))  #creates unique colorbar range for each difference CFAD plot
    # vel_highest_mag = 100  #creates uniform colorbar range across all difference CFAD plots
    # vel_contours = np.linspace(-vel_highest_mag, vel_highest_mag, 41)  #an odd number of intervals (21) guarantees a contour at 0 for a range from -x to x
    
    #grab the case numbers of the 2 cases/case sets being differenced and create appropriate image names
    case1_num = ','.join(map(str, list(case_dict_1.keys())))
    case2_num = ','.join(map(str, list(case_dict_2.keys())))
    cases = case1_num + 'and' + case2_num
    colorbar_label = 'Differential Normalized Frequency [%]'
    #dbz_save_label = 'diff_CFADnorm_Cases_' + cases + '_Ku.png'
    dbz_save_label = save_label_manual + '_Ku.png'
    #vel_save_label = 'diff_CFADnorm_Cases_' + cases + '_Vel.png'
    vel_save_label = save_label_manual + '_Vel.png'

    #plot the Ku-band and Doppler Velocity 2-D difference CFADs
    plot_diffCFAD(dbz_diff_CFAD, dbz_contours, colorbar_label, dbz_save_label, 'Ku-band Reflectivity [dBZ]', dbz_centers_meshgrid, height_centers_meshgrid, dbz_bin_edges, height_bin_edges, case1_num, case2_num, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m, diff_plot_title, dbz_plot = False) 
    # plot_diffCFAD(vel_diff_CFAD, vel_contours, colorbar_label, vel_save_label, 'Mean Doppler Velocity [m/s]', vel_centers_meshgrid, height_vel_centers_meshgrid, vel_bin_edges, height_bin_edges, case1_num, case2_num, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m, dbz_plot = False)
  

def plot_diffCFAD(cfad_array, contours, colorbar_label, save_label, var_label, var_centers_meshgrid, height_centers_meshgrid, var_edges, height_edges, case1_num, case2_num, first_median_height_profile_above1500m, first_medianDBZ_profile_above1500m, first_Q1DBZ_profile_above1500m, first_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m, second_medianDBZ_profile_above1500m, second_Q1DBZ_profile_above1500m, second_Q3DBZ_profile_above1500m, diff_plot_title, dbz_plot = True):
    
    """Plot a contourf difference CFAD given a difference CFAD 2-D array, contour levels, plot/image labels, and variable/height meshgrids"""
    
    #plot the CFAD
    fig, ax = plt.subplots(1,1, figsize=(21,21))
    # cmap = mplc.ListedColormap(['#00429d', '#2855a6', '#3e68af', '#507bb8', '#618fc1', '#73a3ca', '#85b8d3', 
    #                             '#99ccdc', '#b0e0e6', '#cef2f1', '#ffffff', '#ffe5e7', '#ffcbcf', '#ffafb7', 
    #                             '#ff929e', '#f57789', '#e95d76', '#d94364', '#c62a54', '#af1046', '#93003a'])
    
    cmap = mplc.ListedColormap(['#00429d', '#074ca2', '#0e57a8', '#1561ad', '#1d6bb2', '#2475b8', '#2b7fbd', 
                                '#3289c2', '#3994c8', '#409ecd', '#48a8d2', '#4fb3d8', '#56bddd', '#5ec8e3', 
                                '#65d3e8', '#6dddee', '#74e8f4', '#7cf3f9', '#83feff', '#cbffff', '#ffffff', 
                                '#fef3f0', '#fce7e0', '#fbdad0', '#facec1', '#f8c2b1', '#f7b5a1', '#f5a890', 
                                '#f49b7f', '#f28d6e', '#f17f5c', '#ef6f48', '#ed5f33', '#eb4c1b', '#e13f18', 
                                '#d5351e', '#c82b23', '#bc2128', '#af162e', '#a10b34', '#93003a'])
    
    #cs = ax.pcolormesh(dbz_meshgrid, height_meshgrid, cfad_array, cmap = cmap)
    #cs = ax.contour(dbz_centers_meshgrid, height_centers_meshgrid, cfad_array, levels = contours, cmap = cmap, linewidths = 1)
    cs = ax.contourf(var_centers_meshgrid, height_centers_meshgrid / 1000, cfad_array, levels = contours, cmap = cmap)
    if dbz_plot:
        ax.plot(second_medianDBZ_profile_above1500m, second_median_height_profile_above1500m / 1000, color = 'k', linestyle = '-', linewidth = 5, label = 'Case ' + case2_num + ' Median Profile') 
        ax.plot(second_Q1DBZ_profile_above1500m, second_median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5, label = 'Case ' + case2_num + ' Q1 and Q3 Profiles')
        ax.plot(second_Q3DBZ_profile_above1500m, second_median_height_profile_above1500m / 1000, color = 'k', linestyle = '--', linewidth = 5)
        
        ax.plot(first_medianDBZ_profile_above1500m, first_median_height_profile_above1500m / 1000, color = 'darkgoldenrod', linestyle = '-', linewidth = 5, label = 'Case ' + case1_num + ' Median Profile') 
        ax.plot(first_Q1DBZ_profile_above1500m, first_median_height_profile_above1500m / 1000, color = 'darkgoldenrod', linestyle = '--', linewidth = 5, label = 'Case ' + case1_num + ' Q1 and Q3 Profiles')
        ax.plot(first_Q3DBZ_profile_above1500m, first_median_height_profile_above1500m / 1000, color = 'darkgoldenrod', linestyle = '--', linewidth = 5)
        
        ax.legend(loc = 'upper right')
        
    ax.set_ylabel('Altitude [km]', fontsize=30, fontweight = 'bold', labelpad = 20.0)
    ax.set_xlabel(var_label, fontsize=30, fontweight = 'bold', labelpad = 10.0)
    ax.tick_params(length = 15, width = 5, labelsize = 25)
    ax.set_title(diff_plot_title, fontsize=35, fontweight = 'bold')
    ax.set_ylim([(height_edges[0] + 250) / 1000, (height_edges[-1] - 250) / 1000])
    #ax.set_ylim([(height_edges[0] + 250) / 1000, 5.250])    #for CAMP2Ex Case 10 vs. Case 1
    #ax.set_ylim([(height_edges[0] + 250) / 1000, 4.750])   #for CAMP2Ex Case 17 vs. Case 15
    #ax.set_xlim([var_edges[0],var_edges[-1]])
    ax.set_xlim([5,60])  #for Ku-band
    
    #set the colorbar axis
    cax = fig.add_axes([ax.get_position().x0, ax.get_position().y0 - 0.08,
                       ax.get_position().x1-ax.get_position().x0, 0.02])    #Left, bottom, width, height (all [0,1])

    #create the colorbar
    cbar = plt.colorbar(cs, cax = cax, orientation = 'horizontal')
    cbar.ax.tick_params(length = 10, width = 3, labelsize = 23)
    cbar.set_label(label = colorbar_label, fontsize = 30, fontweight = 'bold')
    
    #save the figure
    #plt.savefig(''.join(['/Users/ben/Desktop/CPEX-CV/Coding/CFAD_plots/Case', case1_num, 'and', case2_num, '_Comparison/', save_label]), bbox_inches = 'tight')
    plt.savefig(''.join(['/Users/ben/Desktop/', save_label]), bbox_inches = 'tight')    
    plt.close()


#run the CFAD function to plot the CFADs

#######################################################################################################
#composite Isolated and Organized convective intensity interregional comparisons for tropical West Atlantic (CPEX, CPEX-AW), East Atlantic (CPEX-CV), and Northwest Pacific (CAMP2Ex)

#only stratiform regions (all lifecycle stages)
difference_CFAD(eatl_isolated_case_dict, watl_isolated_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Isolated, Only Stratiform Regions)', 'diffCFAD_WATL_minus_EATL_Isolated_all_lifecycles_Strat')          #WATL minus EATL
difference_CFAD(nwpac_isolated_case_dict, watl_isolated_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Isolated, Only Stratiform Regions)', 'diffCFAD_WATL_minus_NWPAC_Isolated_all_lifecycles_Strat')       #WATL minus NWPAC
difference_CFAD(nwpac_isolated_case_dict, eatl_isolated_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Isolated, Only Stratiform Regions)', 'diffCFAD_EATL_minus_NWPAC_Isolated_all_lifecycles_Strat')       #EATL minus NWPAC
difference_CFAD(eatl_organized_case_dict, watl_organized_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Organized, Only Stratiform Regions)', 'diffCFAD_WATL_minus_EATL_Organized_all_lifecycles_Strat')      #WATL minus EATL
difference_CFAD(nwpac_organized_case_dict, watl_organized_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Organized, Only Stratiform Regions)', 'diffCFAD_WATL_minus_NWPAC_Organized_all_lifecycles_Strat')   #WATL minus NWPAC
difference_CFAD(nwpac_organized_case_dict, eatl_organized_case_dict, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Organized, Only Stratiform Regions)', 'diffCFAD_EATL_minus_NWPAC_Organized_all_lifecycles_Strat')   #EATL minus NWPAC

#only stratiform regions (only if case was at least partly collected during mature lifecycle stage)
difference_CFAD(eatl_isolated_case_dict_growing_or_mature, watl_isolated_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Isolated, Only Stratiform Regions)', 'diffCFAD_WATL_minus_EATL_Isolated_partially_growing_or_mature_lifecycles_Strat')          #WATL minus EATL
difference_CFAD(nwpac_isolated_case_dict_growing_or_mature, watl_isolated_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Isolated, Only Stratiform Regions)', 'diffCFAD_WATL_minus_NWPAC_Isolated_partially_growing_or_mature_lifecycles_Strat')       #WATL minus NWPAC
difference_CFAD(nwpac_isolated_case_dict_growing_or_mature, eatl_isolated_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Isolated, Only Stratiform Regions)', 'diffCFAD_EATL_minus_NWPAC_Isolated_partially_growing_or_mature_lifecycles_Strat')       #EATL minus NWPAC
difference_CFAD(eatl_organized_case_dict_growing_or_mature, watl_organized_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus EATL, Organized, Only Stratiform Regions)', 'diffCFAD_WATL_minus_EATL_Organized_partially_growing_or_mature_lifecycles_Strat')      #WATL minus EATL
difference_CFAD(nwpac_organized_case_dict_growing_or_mature, watl_organized_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (WATL minus NWPAC, Organized, Only Stratiform Regions)', 'diffCFAD_WATL_minus_NWPAC_Organized_partially_growing_or_mature_lifecycles_Strat')   #WATL minus NWPAC
difference_CFAD(nwpac_organized_case_dict_growing_or_mature, eatl_organized_case_dict_growing_or_mature, height_edges, dbz_edges, vel_edges, 'Difference CFAD (EATL minus NWPAC, Organized, Only Stratiform Regions)', 'diffCFAD_EATL_minus_NWPAC_Organized_partially_growing_or_mature_lifecycles_Strat')   #EATL minus NWPAC

